<a href="https://colab.research.google.com/github/Deekshith0625/Machine_Learning/blob/main/PsyStyle_VIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/PsyStyle"

folders = [
    "models",
    "models/vit",
    "models/compatibility",
    "models/preference",
    "models/fusion",
    "data",
    "data/deepfashion1",
    "data/polyvore",
    "data/psychology",
    "wardrobe",
    "outputs",
    "streamlit"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("PsyStyle project structure created!")
print(f"\nProject location:\n{BASE_DIR}")

for folder in folders:
    print("✓", folder)

PsyStyle project structure created!

Project location:
/content/drive/MyDrive/PsyStyle
✓ models
✓ models/vit
✓ models/compatibility
✓ models/preference
✓ models/fusion
✓ data
✓ data/deepfashion1
✓ data/polyvore
✓ data/psychology
✓ wardrobe
✓ outputs
✓ streamlit


In [ ]:
!pip install -q transformers timm scikit-learn pandas numpy pillow matplotlib tqdm

In [ ]:
# Temporary dataset location for this Colab runtime

RUNTIME_DATA = "/content/psystyle_data"

import os
os.makedirs(RUNTIME_DATA, exist_ok=True)

print("Runtime dataset directory:")
print(RUNTIME_DATA)

Runtime dataset directory:
/content/psystyle_data


In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.move(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle authentication configured.")

Kaggle authentication configured.


In [ ]:
!kaggle datasets download \
    -d vishalbsadanand/deepfashion-1 \
    -p /content/psystyle_data \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/vishalbsadanand/deepfashion-1
License(s): unknown
100% 5.70G/5.70G [01:19<00:00, 77.1MB/s]



In [ ]:
import os

DATASETS_DIR = "/content/psystyle_data/datasets"

print("Contents of datasets:\n")

for root, dirs, files in os.walk(DATASETS_DIR):
    level = root.replace(DATASETS_DIR, "").count(os.sep)

    # Don't print thousands of image files
    if level <= 2:
        indent = "    " * level
        print(f"{indent}[FOLDER] {os.path.basename(root)}")

        for file in files[:10]:
            print(f"{indent}    [FILE] {file}")

        if len(files) > 10:
            print(f"{indent}    ... ({len(files)} files total)")

Contents of datasets:

[FOLDER] datasets
    [FOLDER] shape_ann
        [FILE] train_ann_file.txt
        [FILE] val_ann_file.txt
        [FILE] test_ann_file.txt
    [FOLDER] texture_ann
        [FOLDER] test
            [FILE] upper_fused.txt
            [FILE] lower_fused.txt
            [FILE] outer_fused.txt
            [FILE] test_attr_predictor.txt
        [FOLDER] train
            [FILE] upper_fused.txt
            [FILE] lower_fused.txt
            [FILE] outer_fused.txt
        [FOLDER] val
            [FILE] upper_fused.txt
            [FILE] lower_fused.txt
            [FILE] outer_fused.txt
    [FOLDER] train_images
        [FILE] WOMEN-Cardigans-id_00005032-03_4_full.png
        [FILE] MEN-Tees_Tanks-id_00002245-01_4_full.png
        [FILE] WOMEN-Dresses-id_00007087-02_1_front.png
        [FILE] MEN-Tees_Tanks-id_00002070-01_4_full.png
        [FILE] WOMEN-Tees_Tanks-id_00003114-06_4_full.png
        [FILE] WOMEN-Shorts-id_00006431-01_1_front.png
        [FILE] WOMEN-Rom

In [ ]:
import os
from collections import Counter

DATASET_DIR = "/content/psystyle_data/datasets"

TRAIN_DIR = os.path.join(DATASET_DIR, "train_images")
TEST_DIR = os.path.join(DATASET_DIR, "test_images")

def get_category(filename):
    """
    Extract clothing category from DeepFashion filename.

    Example:
    WOMEN-Cardigans-id_00005032-03_4_full.png
    -> Cardigans
    """
    parts = filename.split("-")

    if len(parts) >= 2:
        return parts[1]

    return "Unknown"


# Get image filenames
train_images = [
    f for f in os.listdir(TRAIN_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

test_images = [
    f for f in os.listdir(TEST_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]


# Count categories
train_categories = Counter(get_category(f) for f in train_images)
test_categories = Counter(get_category(f) for f in test_images)


print("DeepFashion 1 Dataset")
print("=" * 40)

print(f"Training images : {len(train_images)}")
print(f"Testing images  : {len(test_images)}")

print("\nTraining categories:")
for category, count in train_categories.most_common():
    print(f"{category:25s} {count}")

print("\nTesting categories:")
for category, count in test_categories.most_common():
    print(f"{category:25s} {count}")

DeepFashion 1 Dataset
Training images : 10335
Testing images  : 1149

Training categories:
Tees_Tanks                2594
Dresses                   1959
Blouses_Shirts            1377
Sweaters                  674
Shorts                    612
Pants                     600
Rompers_Jumpsuits         549
Jackets_Coats             429
Cardigans                 348
Skirts                    340
Sweatshirts_Hoodies       265
Graphic_Tees              190
Denim                     146
Shirts_Polos              103
Jackets_Vests             74
Leggings                  67
Suiting                   8

Testing categories:
Tees_Tanks                291
Dresses                   212
Blouses_Shirts            149
Sweaters                  77
Shorts                    76
Rompers_Jumpsuits         63
Pants                     58
Skirts                    44
Jackets_Coats             40
Cardigans                 33
Sweatshirts_Hoodies       32
Graphic_Tees              26
Denim                     15

In [ ]:
# Usable clothing categories for PsyStyle

CLASS_NAMES = [
    "Tees_Tanks",
    "Dresses",
    "Blouses_Shirts",
    "Sweaters",
    "Shorts",
    "Pants",
    "Rompers_Jumpsuits",
    "Jackets_Coats",
    "Cardigans",
    "Skirts",
    "Sweatshirts_Hoodies",
    "Graphic_Tees",
    "Denim",
    "Shirts_Polos",
    "Jackets_Vests",
    "Leggings"
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index in CLASS_TO_INDEX.items()
}

print("PsyStyle clothing categories")
print("=" * 40)

for index, class_name in INDEX_TO_CLASS.items():
    print(f"{index:2d} -> {class_name}")

print("\nNumber of classes:", len(CLASS_NAMES))

PsyStyle clothing categories
 0 -> Tees_Tanks
 1 -> Dresses
 2 -> Blouses_Shirts
 3 -> Sweaters
 4 -> Shorts
 5 -> Pants
 6 -> Rompers_Jumpsuits
 7 -> Jackets_Coats
 8 -> Cardigans
 9 -> Skirts
10 -> Sweatshirts_Hoodies
11 -> Graphic_Tees
12 -> Denim
13 -> Shirts_Polos
14 -> Jackets_Vests
15 -> Leggings

Number of classes: 16


In [ ]:
import os
import pandas as pd

# Dataset directories
TRAIN_DIR = "/content/psystyle_data/datasets/train_images"
TEST_DIR = "/content/psystyle_data/datasets/test_images"

def get_category(filename):
    """
    Extract category from a DeepFashion filename.

    Example:
    WOMEN-Cardigans-id_00005032-03_4_full.png
    -> Cardigans
    """
    parts = filename.split("-")
    return parts[1] if len(parts) >= 2 else "Unknown"


def build_dataframe(image_dir):
    rows = []

    for filename in os.listdir(image_dir):

        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            continue

        category = get_category(filename)

        # Keep only our 16 selected classes
        if category not in CLASS_TO_INDEX:
            continue

        rows.append({
            "filename": filename,
            "filepath": os.path.join(image_dir, filename),
            "category": category,
            "label": CLASS_TO_INDEX[category]
        })

    return pd.DataFrame(rows)


# Create training and testing DataFrames
train_df = build_dataframe(TRAIN_DIR)
test_df = build_dataframe(TEST_DIR)


print("Dataset prepared")
print("=" * 40)

print("Training images:", len(train_df))
print("Testing images :", len(test_df))

print("\nTraining class distribution:")
print(train_df["category"].value_counts())

print("\nTesting class distribution:")
print(test_df["category"].value_counts())

Dataset prepared
Training images: 10327
Testing images : 1149

Training class distribution:
category
Tees_Tanks             2594
Dresses                1959
Blouses_Shirts         1377
Sweaters                674
Shorts                  612
Pants                   600
Rompers_Jumpsuits       549
Jackets_Coats           429
Cardigans               348
Skirts                  340
Sweatshirts_Hoodies     265
Graphic_Tees            190
Denim                   146
Shirts_Polos            103
Jackets_Vests            74
Leggings                 67
Name: count, dtype: int64

Testing class distribution:
category
Tees_Tanks             291
Dresses                212
Blouses_Shirts         149
Sweaters                77
Shorts                  76
Rompers_Jumpsuits       63
Pants                   58
Skirts                  44
Jackets_Coats           40
Cardigans               33
Sweatshirts_Hoodies     32
Graphic_Tees            26
Denim                   15
Shirts_Polos            11
Jackets_V

In [ ]:
from transformers import ViTImageProcessor

# ViT image processor
processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

print("ViT image processor loaded successfully.")

print("\nImage size:", processor.size)
print("Mean:", processor.image_mean)
print("Std:", processor.image_std)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

ViT image processor loaded successfully.

Image size: SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)
Mean: (0.5, 0.5, 0.5)
Std: (0.5, 0.5, 0.5)


In [ ]:
import torch
from transformers import ViTModel

# Load pretrained Vision Transformer
vit_model = ViTModel.from_pretrained(
    "google/vit-base-patch16-224"
)

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vit_model = vit_model.to(DEVICE)

# We initially use the pretrained ViT as a feature extractor
vit_model.eval()

print("Vision Transformer loaded successfully.")
print("Device:", DEVICE)
print("Hidden size:", vit_model.config.hidden_size)
print("Number of layers:", vit_model.config.num_hidden_layers)
print("Number of attention heads:", vit_model.config.num_attention_heads)

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Vision Transformer loaded successfully.
Device: cuda
Hidden size: 768
Number of layers: 12
Number of attention heads: 12


In [ ]:
from PIL import Image
import torch

# Select one DeepFashion image
sample_path = train_df.iloc[0]["filepath"]
sample_category = train_df.iloc[0]["category"]

# Open image
image = Image.open(sample_path).convert("RGB")

# Process image for ViT
inputs = processor(images=image, return_tensors="pt")

# Move inputs to GPU
inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

# Extract ViT features
with torch.no_grad():
    outputs = vit_model(**inputs)

# CLS token embedding
embedding = outputs.last_hidden_state[:, 0, :]

print("ViT test successful!")
print("=" * 40)
print("Image:", os.path.basename(sample_path))
print("Category:", sample_category)
print("Original image size:", image.size)
print("ViT input shape:", inputs["pixel_values"].shape)
print("Embedding shape:", embedding.shape)

ViT test successful!
Image: WOMEN-Cardigans-id_00005032-03_4_full.png
Category: Cardigans
Original image size: (512, 1024)
ViT input shape: torch.Size([1, 3, 224, 224])
Embedding shape: torch.Size([1, 768])


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

class DeepFashionDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        # Load image
        image = Image.open(row["filepath"]).convert("RGB")

        # Convert image to ViT input format
        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        # Remove batch dimension
        pixel_values = processed["pixel_values"].squeeze(0)

        label = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return pixel_values, label


# Create datasets
train_dataset = DeepFashionDataset(
    train_df,
    processor
)

test_dataset = DeepFashionDataset(
    test_df,
    processor
)


# Batch size
BATCH_SIZE = 16

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("ViT datasets and DataLoaders created!")
print("=" * 45)

print("Training samples :", len(train_dataset))
print("Testing samples  :", len(test_dataset))
print("Batch size       :", BATCH_SIZE)
print("Training batches :", len(train_loader))
print("Testing batches  :", len(test_loader))

ViT datasets and DataLoaders created!
Training samples : 10327
Testing samples  : 1149
Batch size       : 16
Training batches : 646
Testing batches  : 72


In [ ]:
import torch.nn as nn
from transformers import ViTModel


class PsyStyleViT(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        # Pretrained Vision Transformer
        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        # ViT embedding size = 768
        self.classifier = nn.Linear(
            self.vit.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values, return_embedding=False):

        # Pass images through ViT
        outputs = self.vit(
            pixel_values=pixel_values
        )

        # CLS token = first token
        embedding = outputs.last_hidden_state[:, 0, :]

        # Classification
        logits = self.classifier(embedding)

        if return_embedding:
            return logits, embedding

        return logits


# Create model
vit_classifier = PsyStyleViT(
    num_classes=len(CLASS_NAMES)
)

# Move to GPU
vit_classifier = vit_classifier.to(DEVICE)

print("PsyStyle ViT classifier created!")
print("=" * 45)

print("Number of classes :", len(CLASS_NAMES))
print("Embedding size    :", vit_classifier.vit.config.hidden_size)
print("Device             :", DEVICE)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PsyStyle ViT classifier created!
Number of classes : 16
Embedding size    : 768
Device             : cuda


In [ ]:
import torch
import torch.nn as nn

# --------------------------------------------------
# Freeze the ViT backbone initially
# --------------------------------------------------

for param in vit_classifier.vit.parameters():
    param.requires_grad = False


# Keep the classification head trainable
for param in vit_classifier.classifier.parameters():
    param.requires_grad = True


# Loss function
criterion = nn.CrossEntropyLoss()


# Optimizer
optimizer = torch.optim.AdamW(
    vit_classifier.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


# Count parameters
total_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
    if p.requires_grad
)


print("ViT training configuration")
print("=" * 45)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")
print("Frozen parameters    :", f"{total_params - trainable_params:,}")

print("\nLoss function: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning rate: 0.001")

ViT training configuration
Total parameters     : 86,401,552
Trainable parameters : 12,304
Frozen parameters    : 86,389,248

Loss function: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 0.001


In [ ]:
# --------------------------------------------------
# Configure ViT fine-tuning
# --------------------------------------------------

# First freeze the entire ViT
for param in vit_classifier.vit.parameters():
    param.requires_grad = False


# Find the transformer encoder layers
encoder_layers = vit_classifier.vit.encoder.layer \
    if hasattr(vit_classifier.vit, "encoder") \
    else vit_classifier.vit.vit.encoder.layer


# Unfreeze the final 2 transformer layers
for layer in encoder_layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True


# Keep the classification head trainable
for param in vit_classifier.classifier.parameters():
    param.requires_grad = True


# Fine-tuning optimizer
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        vit_classifier.parameters()
    ),
    lr=2e-5,
    weight_decay=1e-4
)


# Count parameters
total_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
    if p.requires_grad
)


print("Fine-tuning configuration")
print("=" * 45)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")
print("Frozen parameters    :", f"{total_params - trainable_params:,}")

print("\nUnfrozen ViT layers: Final 2")
print("Learning rate: 2e-5")

AttributeError: 'ViTModel' object has no attribute 'vit'

In [ ]:
print(vit_classifier.vit)

print("\nTop-level components:")
print(list(vit_classifier.vit._modules.keys()))

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0-11): 12 x ViTLayer(
      (attention): ViTAttention(
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (k_proj): Linear(in_features=768, out_features=768, bias=True)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (o_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (mlp): ViTMLP(
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
      )
      (dropout): Dropout(p=0.0, inplace=Fa

In [ ]:
# --------------------------------------------------
# Configure ViT fine-tuning
# --------------------------------------------------

# Freeze the entire ViT first
for param in vit_classifier.vit.parameters():
    param.requires_grad = False


# Unfreeze the final 2 ViT transformer layers
for layer in vit_classifier.vit.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True


# Keep the 16-class classification head trainable
for param in vit_classifier.classifier.parameters():
    param.requires_grad = True


# Optimizer for fine-tuning
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        vit_classifier.parameters()
    ),
    lr=2e-5,
    weight_decay=1e-4
)


# Count parameters
total_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_classifier.parameters()
    if p.requires_grad
)


print("Fine-tuning configuration")
print("=" * 45)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")
print("Frozen parameters    :", f"{total_params - trainable_params:,}")

print("\nUnfrozen ViT layers: Final 2")
print("Learning rate: 2e-5")

Fine-tuning configuration
Total parameters     : 86,401,552
Trainable parameters : 14,188,048
Frozen parameters    : 72,213,504

Unfrozen ViT layers: Final 2
Learning rate: 2e-5


In [ ]:
from tqdm.auto import tqdm
import torch

NUM_EPOCHS = 3

train_losses = []
train_accuracies = []

test_losses = []
test_accuracies = []


for epoch in range(NUM_EPOCHS):

    # ==========================================
    # TRAINING
    # ==========================================
    vit_classifier.train()

    running_loss = 0.0
    correct = 0
    total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Training"
    )

    for pixel_values, labels in train_bar:

        pixel_values = pixel_values.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        logits = vit_classifier(pixel_values)

        # Calculate loss
        loss = criterion(logits, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * labels.size(0)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct / total:.4f}"
        )

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)


    # ==========================================
    # TESTING / VALIDATION
    # ==========================================
    vit_classifier.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        test_bar = tqdm(
            test_loader,
            desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Testing"
        )

        for pixel_values, labels in test_bar:

            pixel_values = pixel_values.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            # Forward pass
            logits = vit_classifier(pixel_values)

            # Loss
            loss = criterion(
                logits,
                labels
            )

            running_loss += loss.item() * labels.size(0)

            predictions = torch.argmax(
                logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    epoch_test_loss = running_loss / total
    epoch_test_accuracy = correct / total

    test_losses.append(epoch_test_loss)
    test_accuracies.append(epoch_test_accuracy)


    # ==========================================
    # EPOCH RESULTS
    # ==========================================
    print("\n" + "=" * 55)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Accuracy: {epoch_train_accuracy:.4f}"
    )

    print(
        f"Test Loss:  {epoch_test_loss:.4f} | "
        f"Test Accuracy: {epoch_test_accuracy:.4f}"
    )

    print("=" * 55)

Epoch 1/3 - Training:   0%|          | 0/646 [00:00<?, ?it/s]

Epoch 1/3 - Testing:   0%|          | 0/72 [00:00<?, ?it/s]


Epoch 1/3
Train Loss: 1.6281 | Train Accuracy: 0.4890
Test Loss:  1.3032 | Test Accuracy: 0.5796


Epoch 2/3 - Training:   0%|          | 0/646 [00:00<?, ?it/s]

Epoch 2/3 - Testing:   0%|          | 0/72 [00:00<?, ?it/s]


Epoch 2/3
Train Loss: 1.1156 | Train Accuracy: 0.6292
Test Loss:  1.1167 | Test Accuracy: 0.6345


Epoch 3/3 - Training:   0%|          | 0/646 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f14bc4120c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f14bc4120c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 3/3 - Testing:   0%|          | 0/72 [00:00<?, ?it/s]


Epoch 3/3
Train Loss: 0.9252 | Train Accuracy: 0.6906
Test Loss:  1.0397 | Test Accuracy: 0.6397


In [ ]:
import os
import torch

VIT_SAVE_DIR = "/content/drive/MyDrive/PsyStyle/models/vit"

os.makedirs(VIT_SAVE_DIR, exist_ok=True)

VIT_MODEL_PATH = os.path.join(
    VIT_SAVE_DIR,
    "psystyle_vit_fashion.pt"
)

torch.save(
    {
        "model_state_dict": vit_classifier.state_dict(),
        "class_names": CLASS_NAMES,
        "embedding_size": vit_classifier.vit.config.hidden_size,
        "num_classes": len(CLASS_NAMES)
    },
    VIT_MODEL_PATH
)

print("ViT model saved successfully!")
print("\nLocation:")
print(VIT_MODEL_PATH)

print(
    "\nModel size:",
    f"{os.path.getsize(VIT_MODEL_PATH) / (1024**2):.2f} MB"
)

ViT model saved successfully!

Location:
/content/drive/MyDrive/PsyStyle/models/vit/psystyle_vit_fashion.pt

Model size: 329.67 MB


In [ ]:
import os
import subprocess

KAGGLE_POLYVORE_DIR = "/content/psystyle_data/polyvore_kaggle"

os.makedirs(KAGGLE_POLYVORE_DIR, exist_ok=True)

print("Downloading:")
print("firozkiresur/polyvore-metadata")
print()

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        "firozkiresur/polyvore-metadata",
        "-p",
        KAGGLE_POLYVORE_DIR,
        "--unzip"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr)
else:
    print("\nDownload completed successfully!")

print("\nDataset contents:")
for root, dirs, files in os.walk(KAGGLE_POLYVORE_DIR):
    level = root.replace(KAGGLE_POLYVORE_DIR, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{indent}  └── {file} ({size_mb:.2f} MB)")

    # Don't print an enormous directory tree
    if level >= 2:
        dirs[:] = []

Downloading:
firozkiresur/polyvore-metadata

Dataset URL: https://www.kaggle.com/datasets/firozkiresur/polyvore-metadata
License(s): MIT



Download completed successfully!

Dataset contents:
polyvore_kaggle/
  polyvore/
    └── fashion_compatibility_prediction.txt (0.50 MB)
    └── valid_no_dup.json (3.48 MB)
    └── test_no_dup.json (7.14 MB)
    └── train_no_dup.json (43.62 MB)
    └── fill_in_blank_test.json (1.05 MB)


In [ ]:
import os
import json

POLYVORE_KAGGLE_DIR = "/content/psystyle_data/polyvore_kaggle/polyvore"

train_json = os.path.join(POLYVORE_KAGGLE_DIR, "train_no_dup.json")
compat_file = os.path.join(
    POLYVORE_KAGGLE_DIR,
    "fashion_compatibility_prediction.txt"
)

# Load training metadata
with open(train_json, "r") as f:
    train_outfits = json.load(f)

print("Number of training outfits:", len(train_outfits))

# Inspect first outfit
first_outfit = train_outfits[0]

print("\nFirst outfit keys:")
print(first_outfit.keys())

print("\nFirst outfit name:")
print(first_outfit.get("name"))

print("\nNumber of items in first outfit:")
print(len(first_outfit.get("items", [])))

print("\nFirst item:")
print(first_outfit["items"][0])

# Inspect compatibility file
with open(compat_file, "r") as f:
    compatibility_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("\nCompatibility records:", len(compatibility_lines))

print("\nFirst 3 compatibility records:")
for line in compatibility_lines[:3]:
    print(line)

Number of training outfits: 17316

First outfit keys:
dict_keys(['name', 'views', 'items', 'image', 'likes', 'date', 'set_url', 'set_id', 'desc'])

First outfit name:
Casual

Number of items in first outfit:
7

First item:
{'index': 1, 'name': 'mock neck embroidery suede sweatshirt', 'price': 24.0, 'likes': 10, 'image': 'http://img2.polyvoreimg.com/cgi/img-thing?.out=jpg&size=m&tid=194508109', 'categoryid': 4495}

Compatibility records: 7076

First 3 compatibility records:
1 119704139_1 119704139_2 119704139_3 119704139_4 119704139_5
1 119314458_1 119314458_2 119314458_3 119314458_4 119314458_5 119314458_6 119314458_7 119314458_8
1 147361785_1 147361785_2 147361785_3 147361785_4 147361785_5 147361785_6


In [ ]:
import requests

urls = [
    "https://drive.google.com/file/d/13-J4fAPZahauaGycw3j_YvbAHO7tOTW5/view",
    "https://huggingface.co/datasets/Stylique/Polyvore"
]

print("Checking known Polyvore image sources...\n")

for url in urls:
    try:
        response = requests.get(url, timeout=15, allow_redirects=True)

        print("URL:", url)
        print("Status:", response.status_code)
        print("Final URL:", response.url[:150])
        print()

    except Exception as e:
        print("Error:", e)

Checking known Polyvore image sources...

URL: https://drive.google.com/file/d/13-J4fAPZahauaGycw3j_YvbAHO7tOTW5/view
Status: 404
Final URL: https://drive.google.com/file/d/13-J4fAPZahauaGycw3j_YvbAHO7tOTW5/view

URL: https://huggingface.co/datasets/Stylique/Polyvore
Status: 200
Final URL: https://huggingface.co/datasets/Stylique/Polyvore



In [ ]:
# Check the actual files available in the Hugging Face Polyvore dataset

import requests
import json

HF_API = "https://huggingface.co/api/datasets/Stylique/Polyvore/tree/main"

response = requests.get(
    HF_API,
    params={"recursive": "true", "expand": "false"},
    timeout=30
)

print("Status:", response.status_code)

if response.status_code == 200:
    files = response.json()

    print("Total files/directories found:", len(files))
    print("\nRelevant files:")

    for item in files:
        path = item.get("path", "")

        if any(keyword in path.lower() for keyword in [
            "image",
            "metadata",
            "compatibility",
            "category",
            ".tar",
            ".zip"
        ]):
            print(path)
else:
    print("Could not access Hugging Face file listing.")
    print(response.text[:500])

Status: 200
Total files/directories found: 38

Relevant files:
disjoint/compatibility_test.txt
disjoint/compatibility_train.txt
disjoint/compatibility_valid.txt
images.zip
maryland_polyvore_hardneg/compatibility_test.txt
maryland_polyvore_hardneg/compatibility_train.txt
maryland_polyvore_hardneg/compatibility_valid.txt
nondisjoint/compatibility_test.txt
nondisjoint/compatibility_train.txt
nondisjoint/compatibility_valid.txt
polyvore_item_metadata.json


In [ ]:
import os
import requests

POLYVORE_HF_DIR = "/content/psystyle_data/polyvore_hf"
os.makedirs(POLYVORE_HF_DIR, exist_ok=True)

IMAGE_ZIP_PATH = os.path.join(
    POLYVORE_HF_DIR,
    "images.zip"
)

IMAGE_URL = (
    "https://huggingface.co/datasets/"
    "Stylique/Polyvore/resolve/main/images.zip"
)

print("Downloading Polyvore images...")
print("Destination:", IMAGE_ZIP_PATH)

response = requests.get(
    IMAGE_URL,
    stream=True,
    timeout=60
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    total_size = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(IMAGE_ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if total_size:
                    percent = downloaded / total_size * 100
                    print(
                        f"\rDownloaded: {percent:.1f}%",
                        end=""
                    )

    print("\n\nDownload completed!")

    size_gb = os.path.getsize(IMAGE_ZIP_PATH) / (1024 ** 3)
    print(f"File size: {size_gb:.2f} GB")

else:
    print("Download failed.")
    print(response.text[:500])

Destination: /content/psystyle_data/polyvore_hf/images.zip
HTTP status: 200
Downloaded: 100.0%

Download completed!
File size: 2.35 GB


In [ ]:
import zipfile
import os

IMAGE_ZIP_PATH = "/content/psystyle_data/polyvore_hf/images.zip"

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zip_ref:

    files = zip_ref.namelist()

    print("Total files in archive:", len(files))

    print("\nFirst 30 entries:")
    for file in files[:30]:
        print(file)

    # Count image files
    image_files = [
        file for file in files
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("\nTotal image files:", len(image_files))

    # Show a few image filenames
    print("\nSample image files:")
    for file in image_files[:10]:
        print(file)

Total files in archive: 522116

First 30 entries:
images/
__MACOSX/._images
images/136491077.jpg
__MACOSX/images/._136491077.jpg
images/209825984.jpg
__MACOSX/images/._209825984.jpg
images/205588792.jpg
__MACOSX/images/._205588792.jpg
images/210821459.jpg
__MACOSX/images/._210821459.jpg
images/205891892.jpg
__MACOSX/images/._205891892.jpg
images/188062874.jpg
__MACOSX/images/._188062874.jpg
images/138974860.jpg
__MACOSX/images/._138974860.jpg
images/203490521.jpg
__MACOSX/images/._203490521.jpg
images/209438390.jpg
__MACOSX/images/._209438390.jpg
images/107910012.jpg
__MACOSX/images/._107910012.jpg
images/84071287.jpg
__MACOSX/images/._84071287.jpg
images/213942486.jpg
__MACOSX/images/._213942486.jpg
images/95749570.jpg
__MACOSX/images/._95749570.jpg
images/173207348.jpg
__MACOSX/images/._173207348.jpg

Total image files: 522114

Sample image files:
images/136491077.jpg
__MACOSX/images/._136491077.jpg
images/209825984.jpg
__MACOSX/images/._209825984.jpg
images/205588792.jpg
__MACOSX/im

In [ ]:
import os
import re
import zipfile
import requests

# --------------------------------------------------
# Paths
# --------------------------------------------------

POLYVORE_HF_DIR = "/content/psystyle_data/polyvore_hf"
IMAGE_ZIP_PATH = os.path.join(
    POLYVORE_HF_DIR,
    "images.zip"
)

COMPAT_PATH = os.path.join(
    POLYVORE_HF_DIR,
    "compatibility_train.txt"
)

# --------------------------------------------------
# Download compatibility training file
# --------------------------------------------------

COMPAT_URL = (
    "https://huggingface.co/datasets/"
    "Stylique/Polyvore/resolve/main/"
    "disjoint/compatibility_train.txt"
)

print("Downloading compatibility labels...")

response = requests.get(
    COMPAT_URL,
    timeout=60
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError("Could not download compatibility_train.txt")

with open(COMPAT_PATH, "wb") as f:
    f.write(response.content)

print("Saved:", COMPAT_PATH)

# --------------------------------------------------
# Read compatibility item IDs
# --------------------------------------------------

compatibility_ids = set()
compatibility_outfits = 0

with open(COMPAT_PATH, "r") as f:

    for line in f:
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        # First value is the label: 0 or 1
        item_tokens = parts[1:]

        for token in item_tokens:

            # Example:
            # 119704139_1
            #
            # Extract the actual Polyvore item ID
            match = re.match(r"(\d+)", token)

            if match:
                compatibility_ids.add(
                    match.group(1)
                )

        compatibility_outfits += 1

print("\nCompatibility outfits:", compatibility_outfits)
print("Unique compatibility item IDs:", len(compatibility_ids))

# --------------------------------------------------
# Read image IDs directly from ZIP
# --------------------------------------------------

print("\nReading image IDs from ZIP...")
print("(The 2.35 GB archive will NOT be extracted.)")

image_ids = set()

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zip_ref:

    for filename in zip_ref.namelist():

        # Only actual files under images/
        if not filename.startswith("images/"):
            continue

        if not filename.lower().endswith(".jpg"):
            continue

        basename = os.path.basename(filename)

        # 136491077.jpg -> 136491077
        item_id = os.path.splitext(basename)[0]

        if item_id.isdigit():
            image_ids.add(item_id)

print("Unique image IDs:", len(image_ids))

# --------------------------------------------------
# Match IDs
# --------------------------------------------------

matched_ids = compatibility_ids & image_ids
missing_ids = compatibility_ids - image_ids

print("\n" + "=" * 50)
print("ID MATCHING RESULT")
print("=" * 50)

print("Compatibility item IDs :", len(compatibility_ids))
print("Image item IDs         :", len(image_ids))
print("Matched IDs            :", len(matched_ids))
print("Missing IDs            :", len(missing_ids))

match_percentage = (
    len(matched_ids) / len(compatibility_ids) * 100
)

print(f"\nImage coverage: {match_percentage:.2f}%")

print("\nSample matched IDs:")
for item_id in list(matched_ids)[:10]:
    print("✓", item_id)

if missing_ids:
    print("\nSample missing IDs:")
    for item_id in list(missing_ids)[:10]:
        print("✗", item_id)

HTTP status: 200
Saved: /content/psystyle_data/polyvore_hf/compatibility_train.txt

Compatibility outfits: 33990
Unique compatibility item IDs: 16995

Reading image IDs from ZIP...
(The 2.35 GB archive will NOT be extracted.)
Unique image IDs: 261057

ID MATCHING RESULT
Compatibility item IDs : 16995
Image item IDs         : 261057
Matched IDs            : 19
Missing IDs            : 16976

Image coverage: 0.11%

Sample matched IDs:
✓ 204433821
✓ 210498356
✓ 157902642
✓ 204621840
✓ 210366645
✓ 214251937
✓ 198008756
✓ 210134436
✓ 189176899
✓ 196363533

Sample missing IDs:
✗ 224345318
✗ 200891069
✗ 124669694
✗ 225066900
✗ 214325621
✗ 208575558
✗ 221145550
✗ 219188531
✗ 222256080
✗ 182358054


In [ ]:
import json
import os
import zipfile

# --------------------------------------------------
# Existing files
# --------------------------------------------------

POLYVORE_KAGGLE_DIR = "/content/psystyle_data/polyvore_kaggle/polyvore"
POLYVORE_HF_DIR = "/content/psystyle_data/polyvore_hf"

TRAIN_JSON = os.path.join(
    POLYVORE_KAGGLE_DIR,
    "train_no_dup.json"
)

COMPAT_PATH = os.path.join(
    POLYVORE_HF_DIR,
    "compatibility_train.txt"
)

IMAGE_ZIP_PATH = os.path.join(
    POLYVORE_HF_DIR,
    "images.zip"
)

# --------------------------------------------------
# Load Polyvore outfits
# --------------------------------------------------

print("Loading Polyvore outfit metadata...")

with open(TRAIN_JSON, "r") as f:
    outfits = json.load(f)

print("Training outfits loaded:", len(outfits))

# --------------------------------------------------
# Build:
# (set_id, item_index) -> actual item_id
# --------------------------------------------------

item_reference_to_id = {}

for outfit in outfits:

    set_id = str(outfit["set_id"])

    for item in outfit["items"]:

        item_index = str(item["index"])

        # The JSON item image URL contains the real item ID
        image_url = item.get("image", "")

        # Extract tid=XXXXXXXX
        if "tid=" in image_url:
            item_id = image_url.split("tid=")[-1]
            item_id = item_id.split("&")[0]
        else:
            continue

        item_reference_to_id[
            f"{set_id}_{item_index}"
        ] = item_id

print(
    "\nSet-ID + index mappings:",
    len(item_reference_to_id)
)

# --------------------------------------------------
# Read compatibility records
# --------------------------------------------------

compatibility_records = []

with open(COMPAT_PATH, "r") as f:

    for line in f:

        parts = line.strip().split()

        if not parts:
            continue

        label = int(parts[0])
        references = parts[1:]

        compatibility_records.append(
            {
                "label": label,
                "references": references
            }
        )

print(
    "Compatibility records:",
    len(compatibility_records)
)

# --------------------------------------------------
# Convert compatibility references
# to actual Polyvore image IDs
# --------------------------------------------------

compatibility_item_ids = set()

mapped_references = 0
missing_references = 0

for record in compatibility_records:

    for reference in record["references"]:

        if reference in item_reference_to_id:

            item_id = item_reference_to_id[reference]

            compatibility_item_ids.add(
                item_id
            )

            mapped_references += 1

        else:
            missing_references += 1

print("\n" + "=" * 55)
print("COMPATIBILITY → IMAGE ID MAPPING")
print("=" * 55)

print(
    "Unique actual item IDs:",
    len(compatibility_item_ids)
)

print(
    "Mapped item references:",
    mapped_references
)

print(
    "Missing references:",
    missing_references
)

# --------------------------------------------------
# Read actual image IDs from ZIP
# --------------------------------------------------

print("\nReading image IDs from images.zip...")

image_ids = set()

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zip_ref:

    for filename in zip_ref.namelist():

        if not filename.startswith("images/"):
            continue

        if not filename.lower().endswith(".jpg"):
            continue

        basename = os.path.basename(filename)
        item_id = os.path.splitext(basename)[0]

        if item_id.isdigit():
            image_ids.add(item_id)

print(
    "Image IDs in archive:",
    len(image_ids)
)

# --------------------------------------------------
# Correct matching
# --------------------------------------------------

matched_ids = (
    compatibility_item_ids
    & image_ids
)

missing_ids = (
    compatibility_item_ids
    - image_ids
)

coverage = (
    len(matched_ids)
    / len(compatibility_item_ids)
    * 100
)

print("\n" + "=" * 55)
print("FINAL IMAGE COVERAGE")
print("=" * 55)

print(
    "Compatibility item IDs:",
    len(compatibility_item_ids)
)

print(
    "Matched image IDs:",
    len(matched_ids)
)

print(
    "Missing image IDs:",
    len(missing_ids)
)

print(
    f"\nImage coverage: {coverage:.2f}%"
)

print("\nSample matched image IDs:")

for item_id in list(matched_ids)[:10]:
    print("✓", item_id)

if missing_ids:

    print("\nSample missing image IDs:")

    for item_id in list(missing_ids)[:10]:
        print("✗", item_id)

Loading Polyvore outfit metadata...
Training outfits loaded: 17316

Set-ID + index mappings: 114806
Compatibility records: 33990

COMPATIBILITY → IMAGE ID MAPPING
Unique actual item IDs: 3302
Mapped item references: 6744
Missing references: 165126

Reading image IDs from images.zip...
Image IDs in archive: 261057

FINAL IMAGE COVERAGE
Compatibility item IDs: 3302
Matched image IDs: 3067
Missing image IDs: 235

Image coverage: 92.88%

Sample matched image IDs:
✓ 136924987
✓ 192045441
✓ 156078856
✓ 197039419
✓ 139110120
✓ 138924839
✓ 157455095
✓ 173602690
✓ 192452966
✓ 183700989

Sample missing image IDs:
✗ 187442712
✗ 192505764
✗ 195276219
✗ 116706771
✗ 191484495
✗ 196794147
✗ 189462581
✗ 187545266
✗ 190264950
✗ 186859262


In [ ]:
import requests
import json
import os

HF_METADATA_URL = (
    "https://huggingface.co/datasets/"
    "Stylique/Polyvore/resolve/main/"
    "polyvore_item_metadata.json"
)

HF_METADATA_PATH = (
    "/content/psystyle_data/polyvore_hf/"
    "polyvore_item_metadata.json"
)

print("Downloading Polyvore item metadata...")

response = requests.get(
    HF_METADATA_URL,
    timeout=60
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError("Could not download Polyvore metadata.")

with open(HF_METADATA_PATH, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")

with open(HF_METADATA_PATH, "r") as f:
    metadata = json.load(f)

print("\nMetadata type:", type(metadata))

if isinstance(metadata, dict):
    print("Number of metadata entries:", len(metadata))

    first_key = next(iter(metadata))
    print("\nFirst item ID:", first_key)
    print("First item metadata:")
    print(metadata[first_key])

elif isinstance(metadata, list):
    print("Number of metadata entries:", len(metadata))

    print("\nFirst item:")
    print(metadata[0])

HTTP status: 200
Downloaded successfully!

Metadata type: <class 'dict'>
Number of metadata entries: 251008

First item ID: 211990161
First item metadata:
{'url_name': 'neck print chiffon plus size', 'description': '', 'catgeories': '', 'title': '', 'related': '', 'category_id': '15', 'semantic_category': 'tops'}


In [ ]:
import os
import re
import zipfile

# --------------------------------------------------
# Paths
# --------------------------------------------------

HF_DIR = "/content/psystyle_data/polyvore_hf"

COMPAT_PATH = os.path.join(
    HF_DIR,
    "compatibility_train.txt"
)

METADATA_PATH = os.path.join(
    HF_DIR,
    "polyvore_item_metadata.json"
)

IMAGE_ZIP_PATH = os.path.join(
    HF_DIR,
    "images.zip"
)

# --------------------------------------------------
# Load metadata
# --------------------------------------------------

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

metadata_ids = set(metadata.keys())

print("Metadata item IDs:", len(metadata_ids))

# --------------------------------------------------
# Read compatibility file
# --------------------------------------------------

compatibility_ids = set()
total_references = 0

with open(COMPAT_PATH, "r") as f:

    for line in f:

        parts = line.strip().split()

        if not parts:
            continue

        # Skip the label
        for token in parts[1:]:

            total_references += 1

            # Compatibility files use:
            # setID_itemIndex
            #
            # We only want to inspect the first part
            # temporarily.
            compatibility_ids.add(token)

print("\nCompatibility references:", total_references)
print("Unique compatibility references:", len(compatibility_ids))

# --------------------------------------------------
# Inspect format
# --------------------------------------------------

print("\nSample compatibility references:")

for item in list(compatibility_ids)[:20]:
    print(item)

# --------------------------------------------------
# Extract possible numeric IDs
# --------------------------------------------------

numeric_ids = set()

for token in compatibility_ids:

    # Check whether token itself is an item ID
    if token.isdigit():
        numeric_ids.add(token)

print(
    "\nCompatibility references that are direct numeric IDs:",
    len(numeric_ids)
)

# --------------------------------------------------
# Check image archive IDs
# --------------------------------------------------

print("\nReading image IDs from ZIP...")

image_ids = set()

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zip_ref:

    for filename in zip_ref.namelist():

        if not filename.startswith("images/"):
            continue

        if not filename.endswith(".jpg"):
            continue

        item_id = os.path.splitext(
            os.path.basename(filename)
        )[0]

        if item_id.isdigit():
            image_ids.add(item_id)

print("Image IDs:", len(image_ids))

# --------------------------------------------------
# Direct metadata/image consistency
# --------------------------------------------------

metadata_image_matches = metadata_ids & image_ids

print("\n" + "=" * 55)
print("METADATA ↔ IMAGE CHECK")
print("=" * 55)

print(
    "Metadata IDs:",
    len(metadata_ids)
)

print(
    "Image IDs:",
    len(image_ids)
)

print(
    "Metadata IDs with images:",
    len(metadata_image_matches)
)

print(
    f"Coverage: "
    f"{len(metadata_image_matches) / len(metadata_ids) * 100:.2f}%"
)

Metadata item IDs: 251008

Compatibility references: 171870
Unique compatibility references: 85935

Sample compatibility references:
219702465_3
209155826_2
222005420_3
214844533_4
224126691_2
225313454_5
219757916_1
213234224_1
206837342_1
215591933_2
216328294_1
222835426_4
175124241_2
80508256_1
224934224_2
222702092_5
205651306_1
214816372_2
220465766_2
219824960_3

Compatibility references that are direct numeric IDs: 0

Reading image IDs from ZIP...
Image IDs: 261057

METADATA ↔ IMAGE CHECK
Metadata IDs: 251008
Image IDs: 261057
Metadata IDs with images: 251008
Coverage: 100.00%


In [ ]:
import requests

HF_API = "https://huggingface.co/api/datasets/Stylique/Polyvore/tree/main"

response = requests.get(
    HF_API,
    params={
        "recursive": "true",
        "expand": "false"
    },
    timeout=30
)

if response.status_code != 200:
    raise RuntimeError(
        f"Could not access Hugging Face repository: "
        f"{response.status_code}"
    )

files = response.json()

print("All relevant Polyvore metadata files:\n")

for item in files:

    path = item.get("path", "")

    if any(keyword in path.lower() for keyword in [
        "outfit",
        "metadata",
        "category",
        "compatibility"
    ]):

        print(path)

All relevant Polyvore metadata files:

disjoint/compatibility_test.txt
disjoint/compatibility_train.txt
disjoint/compatibility_valid.txt
maryland_polyvore_hardneg/compatibility_test.txt
maryland_polyvore_hardneg/compatibility_train.txt
maryland_polyvore_hardneg/compatibility_valid.txt
nondisjoint/compatibility_test.txt
nondisjoint/compatibility_train.txt
nondisjoint/compatibility_valid.txt
polyvore_item_metadata.json
polyvore_outfit_titles.json


In [ ]:
import requests
import json
import os

HF_DIR = "/content/psystyle_data/polyvore_hf"

OUTFIT_URL = (
    "https://huggingface.co/datasets/"
    "Stylique/Polyvore/resolve/main/"
    "polyvore_outfit_titles.json"
)

OUTFIT_PATH = os.path.join(
    HF_DIR,
    "polyvore_outfit_titles.json"
)

print("Downloading polyvore_outfit_titles.json...")

response = requests.get(
    OUTFIT_URL,
    timeout=60
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError("Could not download outfit titles.")

with open(OUTFIT_PATH, "wb") as f:
    f.write(response.content)

with open(OUTFIT_PATH, "r") as f:
    outfit_titles = json.load(f)

print("\nType:", type(outfit_titles))

if isinstance(outfit_titles, dict):

    print("Number of entries:", len(outfit_titles))

    first_key = next(iter(outfit_titles))

    print("\nFirst key:", first_key)
    print("First value:")
    print(outfit_titles[first_key])

elif isinstance(outfit_titles, list):

    print("Number of entries:", len(outfit_titles))

    print("\nFirst entry:")
    print(outfit_titles[0])

HTTP status: 200

Type: <class 'dict'>
Number of entries: 68306

First key: 219779031
First value:
{'url_name': 'spring trend colored denim', 'title': 'Denim Jumpers'}


In [ ]:
import os
import requests
import json

HF_DIR = "/content/psystyle_data/polyvore_hf"

TRAIN_URL = (
    "https://huggingface.co/datasets/"
    "Stylique/Polyvore/resolve/main/"
    "disjoint/train.json"
)

TRAIN_PATH = os.path.join(
    HF_DIR,
    "disjoint_train.json"
)

print("Downloading disjoint/train.json...")

response = requests.get(
    TRAIN_URL,
    timeout=60
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        "Could not download disjoint/train.json"
    )

with open(TRAIN_PATH, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")

with open(TRAIN_PATH, "r") as f:
    polyvore_train = json.load(f)

print("\nType:", type(polyvore_train))
print("Number of outfits:", len(polyvore_train))

print("\nFirst outfit:")

if isinstance(polyvore_train, list):
    print(polyvore_train[0])

elif isinstance(polyvore_train, dict):
    first_key = next(iter(polyvore_train))
    print("Set ID:", first_key)
    print(polyvore_train[first_key])

HTTP status: 200
Downloaded successfully!

Type: <class 'list'>
Number of outfits: 16995

First outfit:
{'items': [{'item_id': '132621870', 'index': 1}, {'item_id': '153967122', 'index': 2}, {'item_id': '171169800', 'index': 3}, {'item_id': '162799044', 'index': 4}, {'item_id': '172538912', 'index': 5}, {'item_id': '172312529', 'index': 6}], 'set_id': '199244701'}


In [ ]:
import json
import os

HF_DIR = "/content/psystyle_data/polyvore_hf"

TRAIN_JSON = os.path.join(
    HF_DIR,
    "disjoint_train.json"
)

COMPAT_PATH = os.path.join(
    HF_DIR,
    "compatibility_train.txt"
)

# --------------------------------------------------
# Load official disjoint training outfits
# --------------------------------------------------

with open(TRAIN_JSON, "r") as f:
    polyvore_train = json.load(f)

print("Training outfits:", len(polyvore_train))

# --------------------------------------------------
# Build:
# (set_id, index) -> item_id
# --------------------------------------------------

reference_to_item = {}

for outfit in polyvore_train:

    set_id = str(outfit["set_id"])

    for item in outfit["items"]:

        index = str(item["index"])
        item_id = str(item["item_id"])

        reference = f"{set_id}_{index}"

        reference_to_item[reference] = item_id

print(
    "Available set/index mappings:",
    len(reference_to_item)
)

# --------------------------------------------------
# Read compatibility training file
# --------------------------------------------------

compatibility_samples = []

with open(COMPAT_PATH, "r") as f:

    for line in f:

        parts = line.strip().split()

        if not parts:
            continue

        label = int(parts[0])
        references = parts[1:]

        compatibility_samples.append(
            {
                "label": label,
                "references": references
            }
        )

print(
    "Compatibility samples:",
    len(compatibility_samples)
)

# --------------------------------------------------
# Convert references → actual item IDs
# --------------------------------------------------

usable_samples = []
missing_reference_count = 0

for sample in compatibility_samples:

    item_ids = []

    valid = True

    for reference in sample["references"]:

        if reference not in reference_to_item:

            valid = False
            missing_reference_count += 1
            break

        item_ids.append(
            reference_to_item[reference]
        )

    if valid:

        usable_samples.append(
            {
                "label": sample["label"],
                "item_ids": item_ids
            }
        )

print("\n" + "=" * 55)
print("COMPATIBILITY DATASET")
print("=" * 55)

print(
    "Original samples:",
    len(compatibility_samples)
)

print(
    "Usable samples:",
    len(usable_samples)
)

print(
    "Samples with missing references:",
    missing_reference_count
)

# --------------------------------------------------
# Unique images actually needed
# --------------------------------------------------

required_item_ids = set()

for sample in usable_samples:

    required_item_ids.update(
        sample["item_ids"]
    )

print(
    "\nUnique Polyvore images required:",
    len(required_item_ids)
)

# --------------------------------------------------
# Label distribution
# --------------------------------------------------

positive = sum(
    sample["label"] == 1
    for sample in usable_samples
)

negative = sum(
    sample["label"] == 0
    for sample in usable_samples
)

print("\nLabel distribution:")
print("Compatible   (1):", positive)
print("Incompatible (0):", negative)

# --------------------------------------------------
# Save required IDs
# --------------------------------------------------

REQUIRED_IDS_PATH = os.path.join(
    HF_DIR,
    "required_training_item_ids.json"
)

with open(REQUIRED_IDS_PATH, "w") as f:
    json.dump(
        sorted(required_item_ids),
        f,
        indent=2
    )

print(
    "\nRequired ID list saved to:"
)

print(REQUIRED_IDS_PATH)

Training outfits: 16995
Available set/index mappings: 85935
Compatibility samples: 33990

COMPATIBILITY DATASET
Original samples: 33990
Usable samples: 33990
Samples with missing references: 0

Unique Polyvore images required: 71967

Label distribution:
Compatible   (1): 16995
Incompatible (0): 16995

Required ID list saved to:
/content/psystyle_data/polyvore_hf/required_training_item_ids.json


In [ ]:
import os
import json
import zipfile

HF_DIR = "/content/psystyle_data/polyvore_hf"

IMAGE_ZIP_PATH = os.path.join(
    HF_DIR,
    "images.zip"
)

REQUIRED_IDS_PATH = os.path.join(
    HF_DIR,
    "required_training_item_ids.json"
)

EXTRACT_DIR = os.path.join(
    HF_DIR,
    "compatibility_images"
)

os.makedirs(EXTRACT_DIR, exist_ok=True)

# --------------------------------------------------
# Load required item IDs
# --------------------------------------------------

with open(REQUIRED_IDS_PATH, "r") as f:
    required_ids = set(json.load(f))

print("Required images:", len(required_ids))

# --------------------------------------------------
# Extract only required images
# --------------------------------------------------

extracted = 0
missing = []

print("\nExtracting required Polyvore images...")

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zip_ref:

    for item_id in required_ids:

        filename = f"images/{item_id}.jpg"

        output_path = os.path.join(
            EXTRACT_DIR,
            f"{item_id}.jpg"
        )

        try:

            with zip_ref.open(filename) as source:
                with open(output_path, "wb") as target:
                    target.write(source.read())

            extracted += 1

            if extracted % 1000 == 0:
                print(
                    f"Extracted: "
                    f"{extracted}/{len(required_ids)}"
                )

        except KeyError:

            missing.append(item_id)

print("\n" + "=" * 55)
print("EXTRACTION COMPLETE")
print("=" * 55)

print("Required images :", len(required_ids))
print("Extracted images:", extracted)
print("Missing images  :", len(missing))

print("\nImages saved to:")
print(EXTRACT_DIR)

if missing:
    print("\nFirst missing IDs:")
    print(missing[:20])

Required images: 71967

Extracting required Polyvore images...
Extracted: 1000/71967
Extracted: 2000/71967
Extracted: 3000/71967
Extracted: 4000/71967
Extracted: 5000/71967
Extracted: 6000/71967
Extracted: 7000/71967
Extracted: 8000/71967
Extracted: 9000/71967
Extracted: 10000/71967
Extracted: 11000/71967
Extracted: 12000/71967
Extracted: 13000/71967
Extracted: 14000/71967
Extracted: 15000/71967
Extracted: 16000/71967
Extracted: 17000/71967
Extracted: 18000/71967
Extracted: 19000/71967
Extracted: 20000/71967
Extracted: 21000/71967
Extracted: 22000/71967
Extracted: 23000/71967
Extracted: 24000/71967
Extracted: 25000/71967
Extracted: 26000/71967
Extracted: 27000/71967
Extracted: 28000/71967
Extracted: 29000/71967
Extracted: 30000/71967
Extracted: 31000/71967
Extracted: 32000/71967
Extracted: 33000/71967
Extracted: 34000/71967
Extracted: 35000/71967
Extracted: 36000/71967
Extracted: 37000/71967
Extracted: 38000/71967
Extracted: 39000/71967
Extracted: 40000/71967
Extracted: 41000/71967
Ext

In [ ]:
import os
import torch
import torch.nn as nn
from transformers import ViTModel, ViTImageProcessor

# --------------------------------------------------
# Device
# --------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# --------------------------------------------------
# Paths
# --------------------------------------------------

VIT_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "PsyStyle/models/vit/"
    "psystyle_vit_fashion.pt"
)

# --------------------------------------------------
# Load processor
# --------------------------------------------------

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# --------------------------------------------------
# Define the same ViT architecture
# --------------------------------------------------

class PsyStyleViT(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.classifier = nn.Linear(
            self.vit.config.hidden_size,
            num_classes
        )

    def forward(
        self,
        pixel_values,
        return_embedding=False
    ):

        outputs = self.vit(
            pixel_values=pixel_values
        )

        embedding = (
            outputs.last_hidden_state[:, 0, :]
        )

        logits = self.classifier(
            embedding
        )

        if return_embedding:
            return logits, embedding

        return logits


# --------------------------------------------------
# Load saved checkpoint
# --------------------------------------------------

checkpoint = torch.load(
    VIT_MODEL_PATH,
    map_location=DEVICE
)

CLASS_NAMES = checkpoint["class_names"]

vit_classifier = PsyStyleViT(
    num_classes=len(CLASS_NAMES)
)

vit_classifier.load_state_dict(
    checkpoint["model_state_dict"]
)

vit_classifier = vit_classifier.to(DEVICE)

vit_classifier.eval()

# --------------------------------------------------
# Verify
# --------------------------------------------------

print("\nViT loaded successfully!")
print("Number of classes:", len(CLASS_NAMES))
print("Embedding size:", checkpoint["embedding_size"])
print("Model device:", DEVICE)

Device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



ViT loaded successfully!
Number of classes: 16
Embedding size: 768
Model device: cuda


In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# Paths
# --------------------------------------------------

POLYVORE_IMAGE_DIR = (
    "/content/psystyle_data/"
    "polyvore_hf/compatibility_images"
)

EMBEDDING_DIR = (
    "/content/drive/MyDrive/"
    "PsyStyle/models/compatibility"
)

os.makedirs(EMBEDDING_DIR, exist_ok=True)

EMBEDDING_PATH = os.path.join(
    EMBEDDING_DIR,
    "polyvore_vit_embeddings.npz"
)

# --------------------------------------------------
# Image list
# --------------------------------------------------

image_files = [
    filename
    for filename in os.listdir(POLYVORE_IMAGE_DIR)
    if filename.lower().endswith(".jpg")
]

image_files.sort()

print("Polyvore images:", len(image_files))

# --------------------------------------------------
# Batch settings
# --------------------------------------------------

BATCH_SIZE = 32

all_embeddings = []
all_item_ids = []

# --------------------------------------------------
# Generate embeddings
# --------------------------------------------------

vit_classifier.eval()

with torch.no_grad():

    for start in tqdm(
        range(
            0,
            len(image_files),
            BATCH_SIZE
        ),
        desc="Generating ViT embeddings"
    ):

        batch_files = image_files[
            start:start + BATCH_SIZE
        ]

        images = []
        valid_ids = []

        for filename in batch_files:

            path = os.path.join(
                POLYVORE_IMAGE_DIR,
                filename
            )

            try:

                image = Image.open(
                    path
                ).convert("RGB")

                images.append(image)

                item_id = os.path.splitext(
                    filename
                )[0]

                valid_ids.append(item_id)

            except Exception as e:

                print(
                    "\nSkipping:",
                    filename,
                    "| Error:",
                    e
                )

        if not images:
            continue

        # Process batch
        processed = processor(
            images=images,
            return_tensors="pt"
        )

        pixel_values = (
            processed["pixel_values"]
            .to(DEVICE)
        )

        # Get 768-D ViT embeddings
        _, embeddings = vit_classifier(
            pixel_values,
            return_embedding=True
        )

        embeddings = (
            embeddings
            .cpu()
            .numpy()
        )

        all_embeddings.append(
            embeddings
        )

        all_item_ids.extend(
            valid_ids
        )

# --------------------------------------------------
# Combine all batches
# --------------------------------------------------

all_embeddings = np.concatenate(
    all_embeddings,
    axis=0
)

all_item_ids = np.array(
    all_item_ids
)

print("\n" + "=" * 55)
print("EMBEDDING GENERATION COMPLETE")
print("=" * 55)

print(
    "Number of embeddings:",
    len(all_embeddings)
)

print(
    "Embedding shape:",
    all_embeddings.shape
)

print(
    "Item IDs:",
    len(all_item_ids)
)

# --------------------------------------------------
# Save
# --------------------------------------------------

np.savez_compressed(
    EMBEDDING_PATH,
    embeddings=all_embeddings,
    item_ids=all_item_ids
)

print("\nSaved embeddings to:")
print(EMBEDDING_PATH)

file_size = (
    os.path.getsize(EMBEDDING_PATH)
    / (1024 ** 2)
)

print(
    f"\nFile size: {file_size:.2f} MB"
)

Polyvore images: 71967


Generating ViT embeddings:   0%|          | 0/2249 [00:00<?, ?it/s]


EMBEDDING GENERATION COMPLETE
Number of embeddings: 71967
Embedding shape: (71967, 768)
Item IDs: 71967

Saved embeddings to:
/content/drive/MyDrive/PsyStyle/models/compatibility/polyvore_vit_embeddings.npz

File size: 195.73 MB


In [ ]:
import os
import numpy as np

# --------------------------------------------------
# Path to saved Polyvore embeddings
# --------------------------------------------------

EMBEDDING_PATH = (
    "/content/drive/MyDrive/"
    "PsyStyle/models/compatibility/"
    "polyvore_vit_embeddings.npz"
)

# --------------------------------------------------
# Load embeddings
# --------------------------------------------------

data = np.load(EMBEDDING_PATH)

polyvore_embeddings = data["embeddings"]
polyvore_item_ids = data["item_ids"]

# --------------------------------------------------
# Create item_id → embedding lookup
# --------------------------------------------------

item_to_embedding = {
    item_id: embedding
    for item_id, embedding
    in zip(polyvore_item_ids, polyvore_embeddings)
}

# --------------------------------------------------
# Verification
# --------------------------------------------------

print("=" * 55)
print("POLYVORE EMBEDDINGS LOADED")
print("=" * 55)

print("Number of embeddings:", len(polyvore_embeddings))
print("Embedding shape:", polyvore_embeddings.shape)
print("Number of item IDs:", len(polyvore_item_ids))
print("Lookup entries:", len(item_to_embedding))

# Check one example
sample_item_id = polyvore_item_ids[0]

print("\nExample item ID:", sample_item_id)
print(
    "Example embedding shape:",
    item_to_embedding[sample_item_id].shape
)

POLYVORE EMBEDDINGS LOADED
Number of embeddings: 71967
Embedding shape: (71967, 768)
Number of item IDs: 71967
Lookup entries: 71967

Example item ID: 100015595
Example embedding shape: (768,)


In [ ]:
import json
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------

POLYVORE_DIR = (
    "/content/psystyle_data/"
    "polyvore_hf"
)

TRAIN_JSON_PATH = os.path.join(
    POLYVORE_DIR,
    "disjoint_train.json"
)

COMPATIBILITY_PATH = os.path.join(
    POLYVORE_DIR,
    "compatibility_train.txt"
)

# --------------------------------------------------
# Load outfit → item mapping
# --------------------------------------------------

with open(TRAIN_JSON_PATH, "r") as f:
    train_outfits = json.load(f)

print("Training outfits:", len(train_outfits))

# --------------------------------------------------
# Create:
# (set_id, item_index) → item_id
# --------------------------------------------------

set_index_to_item = {}

for outfit in train_outfits:

    set_id = str(outfit["set_id"])

    for item in outfit["items"]:

        item_index = str(item["index"])
        item_id = str(item["item_id"])

        key = f"{set_id}_{item_index}"

        set_index_to_item[key] = item_id

print(
    "Set/index mappings:",
    len(set_index_to_item)
)

# --------------------------------------------------
# Load compatibility samples
# --------------------------------------------------

compatibility_samples = []

with open(COMPATIBILITY_PATH, "r") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        parts = line.split()

        label = int(parts[0])
        references = parts[1:]

        compatibility_samples.append({
            "label": label,
            "references": references
        })

print(
    "Compatibility samples:",
    len(compatibility_samples)
)

# --------------------------------------------------
# Convert references → actual item IDs
# --------------------------------------------------

mapped_samples = []

missing_mappings = 0

for sample in compatibility_samples:

    item_ids = []

    valid = True

    for reference in sample["references"]:

        if reference not in set_index_to_item:
            valid = False
            missing_mappings += 1
            break

        item_ids.append(
            set_index_to_item[reference]
        )

    if valid:
        mapped_samples.append({
            "item_ids": item_ids,
            "label": sample["label"]
        })

# --------------------------------------------------
# Check whether all items have ViT embeddings
# --------------------------------------------------

missing_embeddings = 0

for sample in mapped_samples:

    for item_id in sample["item_ids"]:

        if item_id not in item_to_embedding:
            missing_embeddings += 1

# --------------------------------------------------
# Results
# --------------------------------------------------

print("\n" + "=" * 55)
print("COMPATIBILITY DATA MAPPING COMPLETE")
print("=" * 55)

print(
    "Original samples:",
    len(compatibility_samples)
)

print(
    "Successfully mapped samples:",
    len(mapped_samples)
)

print(
    "Samples with missing set/index mapping:",
    missing_mappings
)

print(
    "Missing ViT embeddings:",
    missing_embeddings
)

# --------------------------------------------------
# Show one example
# --------------------------------------------------

if mapped_samples:

    example = mapped_samples[0]

    print("\nExample sample:")
    print("Label:", example["label"])
    print("Number of items:", len(example["item_ids"]))
    print("Item IDs:", example["item_ids"])

    first_item = example["item_ids"][0]

    print(
        "\nFirst item embedding shape:",
        item_to_embedding[first_item].shape
    )

Training outfits: 16995
Set/index mappings: 85935
Compatibility samples: 33990

COMPATIBILITY DATA MAPPING COMPLETE
Original samples: 33990
Successfully mapped samples: 33990
Samples with missing set/index mapping: 0
Missing ViT embeddings: 0

Example sample:
Label: 1
Number of items: 6
Item IDs: ['132621870', '153967122', '171169800', '162799044', '172538912', '172312529']

First item embedding shape: (768,)


In [ ]:
import torch
import numpy as np

# --------------------------------------------------
# Convert mapped samples into embedding sequences
# --------------------------------------------------

X_embeddings = []
y_labels = []

for sample in mapped_samples:

    item_embeddings = []

    for item_id in sample["item_ids"]:

        embedding = item_to_embedding[item_id]

        item_embeddings.append(embedding)

    # Shape:
    # number_of_items × 768
    item_embeddings = np.stack(
        item_embeddings
    )

    X_embeddings.append(
        item_embeddings
    )

    y_labels.append(
        sample["label"]
    )

# --------------------------------------------------
# Check outfit sizes
# --------------------------------------------------

outfit_lengths = [
    len(sample["item_ids"])
    for sample in mapped_samples
]

print("=" * 55)
print("COMPATIBILITY SAMPLES PREPARED")
print("=" * 55)

print("Total samples:", len(X_embeddings))

print(
    "Minimum items in an outfit:",
    min(outfit_lengths)
)

print(
    "Maximum items in an outfit:",
    max(outfit_lengths)
)

print(
    "Average items per outfit:",
    round(np.mean(outfit_lengths), 2)
)

print(
    "First sample shape:",
    X_embeddings[0].shape
)

print(
    "First sample label:",
    y_labels[0]
)

COMPATIBILITY SAMPLES PREPARED
Total samples: 33990
Minimum items in an outfit: 2
Maximum items in an outfit: 16
Average items per outfit: 5.06
First sample shape: (6, 768)
First sample label: 1


In [ ]:
import torch
import torch.nn as nn

# --------------------------------------------------
# Compatibility Network
# --------------------------------------------------

class CompatibilityNetwork(nn.Module):

    def __init__(self, embedding_dim=768):

        super().__init__()

        # Encode each clothing item
        self.item_encoder = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Learn compatibility from the
        # complete outfit representation
        self.compatibility_network = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1)
        )

    def masked_mean_pool(
        self,
        item_embeddings,
        mask
    ):
        """
        item_embeddings:
            [batch, max_items, 256]

        mask:
            [batch, max_items]
        """

        mask = mask.unsqueeze(-1).float()

        masked_embeddings = (
            item_embeddings * mask
        )

        summed = masked_embeddings.sum(
            dim=1
        )

        count = mask.sum(
            dim=1
        ).clamp(min=1)

        return summed / count

    def forward(
        self,
        embeddings,
        mask
    ):

        # Encode every clothing item
        encoded_items = self.item_encoder(
            embeddings
        )

        # Combine variable number of clothing
        # items into one outfit representation
        outfit_embedding = self.masked_mean_pool(
            encoded_items,
            mask
        )

        # Predict compatibility
        logits = self.compatibility_network(
            outfit_embedding
        )

        return logits.squeeze(-1)


# --------------------------------------------------
# Create model
# --------------------------------------------------

compatibility_model = CompatibilityNetwork(
    embedding_dim=768
)

compatibility_model = compatibility_model.to(
    DEVICE
)

print("=" * 55)
print("COMPATIBILITY NETWORK CREATED")
print("=" * 55)

print(
    "Input embedding size:",
    768
)

print(
    "Item representation size:",
    256
)

print(
    "Outfit representation size:",
    256
)

print(
    "Model device:",
    DEVICE
)

# --------------------------------------------------
# Parameter count
# --------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in compatibility_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in compatibility_model.parameters()
    if p.requires_grad
)

print(
    "\nTotal parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

COMPATIBILITY NETWORK CREATED
Input embedding size: 768
Item representation size: 256
Outfit representation size: 256
Model device: cuda

Total parameters: 229,889
Trainable parameters: 229,889


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# Compatibility Dataset
# --------------------------------------------------

class CompatibilityDataset(Dataset):

    def __init__(
        self,
        embeddings,
        labels,
        max_items=16
    ):

        self.embeddings = embeddings
        self.labels = labels
        self.max_items = max_items

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, index):

        outfit = self.embeddings[index]
        label = self.labels[index]

        # Number of actual clothing items
        num_items = outfit.shape[0]

        # Create zero-padded tensor
        padded = torch.zeros(
            self.max_items,
            768,
            dtype=torch.float32
        )

        # Create mask
        mask = torch.zeros(
            self.max_items,
            dtype=torch.bool
        )

        # Copy actual embeddings
        padded[:num_items] = torch.tensor(
            outfit,
            dtype=torch.float32
        )

        # Mark real items as True
        mask[:num_items] = True

        return (
            padded,
            mask,
            torch.tensor(
                label,
                dtype=torch.float32
            )
        )


# --------------------------------------------------
# Create dataset
# --------------------------------------------------

compatibility_dataset = CompatibilityDataset(
    X_embeddings,
    y_labels,
    max_items=16
)

# --------------------------------------------------
# Create DataLoader
# --------------------------------------------------

COMPATIBILITY_BATCH_SIZE = 64

compatibility_loader = DataLoader(
    compatibility_dataset,
    batch_size=COMPATIBILITY_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

# --------------------------------------------------
# Verify one batch
# --------------------------------------------------

batch_embeddings, batch_masks, batch_labels = next(
    iter(compatibility_loader)
)

print("=" * 55)
print("COMPATIBILITY DATASET READY")
print("=" * 55)

print(
    "Total samples:",
    len(compatibility_dataset)
)

print(
    "Batch size:",
    COMPATIBILITY_BATCH_SIZE
)

print(
    "Embedding batch shape:",
    batch_embeddings.shape
)

print(
    "Mask batch shape:",
    batch_masks.shape
)

print(
    "Label batch shape:",
    batch_labels.shape
)

print(
    "\nFirst outfit:",
    batch_embeddings[0].shape
)

print(
    "Number of real items in first outfit:",
    batch_masks[0].sum().item()
)

print(
    "First label:",
    batch_labels[0].item()
)

COMPATIBILITY DATASET READY
Total samples: 33990
Batch size: 64
Embedding batch shape: torch.Size([64, 16, 768])
Mask batch shape: torch.Size([64, 16])
Label batch shape: torch.Size([64])

First outfit: torch.Size([16, 768])
Number of real items in first outfit: 5
First label: 1.0


In [ ]:
from torch.utils.data import random_split

# --------------------------------------------------
# Reproducible train/validation split
# --------------------------------------------------

TOTAL_SAMPLES = len(compatibility_dataset)

TRAIN_SIZE = int(
    0.80 * TOTAL_SAMPLES
)

VALIDATION_SIZE = (
    TOTAL_SAMPLES - TRAIN_SIZE
)

generator = torch.Generator().manual_seed(42)

train_dataset, validation_dataset = random_split(
    compatibility_dataset,
    [TRAIN_SIZE, VALIDATION_SIZE],
    generator=generator
)

# --------------------------------------------------
# DataLoaders
# --------------------------------------------------

COMPATIBILITY_BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=COMPATIBILITY_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=COMPATIBILITY_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# --------------------------------------------------
# Display split
# --------------------------------------------------

print("=" * 55)
print("TRAIN / VALIDATION SPLIT")
print("=" * 55)

print(
    "Total samples:",
    TOTAL_SAMPLES
)

print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(validation_dataset)
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

TRAIN / VALIDATION SPLIT
Total samples: 33990
Training samples: 27192
Validation samples: 6798
Training batches: 425
Validation batches: 107


In [ ]:
import torch
import torch.nn as nn

# --------------------------------------------------
# Loss function
# --------------------------------------------------

criterion = nn.BCEWithLogitsLoss()

# --------------------------------------------------
# Optimizer
# --------------------------------------------------

optimizer = torch.optim.AdamW(
    compatibility_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

NUM_EPOCHS = 15

print("=" * 55)
print("COMPATIBILITY TRAINING CONFIGURATION")
print("=" * 55)

print(
    "Loss function:",
    "BCEWithLogitsLoss"
)

print(
    "Optimizer:",
    "AdamW"
)

print(
    "Learning rate:",
    1e-3
)

print(
    "Weight decay:",
    1e-4
)

print(
    "Epochs:",
    NUM_EPOCHS
)

print(
    "Device:",
    DEVICE
)

COMPATIBILITY TRAINING CONFIGURATION
Loss function: BCEWithLogitsLoss
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.0001
Epochs: 15
Device: cuda


In [ ]:
import os
import torch
from tqdm.auto import tqdm

# --------------------------------------------------
# Save path
# --------------------------------------------------

COMPATIBILITY_SAVE_DIR = (
    "/content/drive/MyDrive/"
    "PsyStyle/models/compatibility"
)

os.makedirs(
    COMPATIBILITY_SAVE_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    COMPATIBILITY_SAVE_DIR,
    "psystyle_compatibility.pt"
)

# --------------------------------------------------
# Training history
# --------------------------------------------------

train_losses = []
train_accuracies = []

validation_losses = []
validation_accuracies = []

best_validation_accuracy = 0.0

# --------------------------------------------------
# Training loop
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    # ==============================================
    # TRAINING
    # ==============================================

    compatibility_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for embeddings, mask, labels in progress_bar:

        embeddings = embeddings.to(
            DEVICE,
            non_blocking=True
        )

        mask = mask.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        logits = compatibility_model(
            embeddings,
            mask
        )

        # Calculate loss
        loss = criterion(
            logits,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # ------------------------------------------
        # Statistics
        # ------------------------------------------

        running_loss += (
            loss.item() *
            labels.size(0)
        )

        probabilities = torch.sigmoid(
            logits
        )

        predictions = (
            probabilities >= 0.5
        ).float()

        correct += (
            (predictions == labels)
            .sum()
            .item()
        )

        total += labels.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_train_loss = (
        running_loss / total
    )

    epoch_train_accuracy = (
        correct / total
    )

    # ==============================================
    # VALIDATION
    # ==============================================

    compatibility_model.eval()

    validation_running_loss = 0.0
    validation_correct = 0
    validation_total = 0

    with torch.no_grad():

        for embeddings, mask, labels in validation_loader:

            embeddings = embeddings.to(
                DEVICE,
                non_blocking=True
            )

            mask = mask.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            logits = compatibility_model(
                embeddings,
                mask
            )

            loss = criterion(
                logits,
                labels
            )

            validation_running_loss += (
                loss.item() *
                labels.size(0)
            )

            probabilities = torch.sigmoid(
                logits
            )

            predictions = (
                probabilities >= 0.5
            ).float()

            validation_correct += (
                (predictions == labels)
                .sum()
                .item()
            )

            validation_total += (
                labels.size(0)
            )

    epoch_validation_loss = (
        validation_running_loss /
        validation_total
    )

    epoch_validation_accuracy = (
        validation_correct /
        validation_total
    )

    # ==============================================
    # Store history
    # ==============================================

    train_losses.append(
        epoch_train_loss
    )

    train_accuracies.append(
        epoch_train_accuracy
    )

    validation_losses.append(
        epoch_validation_loss
    )

    validation_accuracies.append(
        epoch_validation_accuracy
    )

    # ==============================================
    # Save best model
    # ==============================================

    if (
        epoch_validation_accuracy
        > best_validation_accuracy
    ):

        best_validation_accuracy = (
            epoch_validation_accuracy
        )

        torch.save(
            {
                "model_state_dict":
                    compatibility_model.state_dict(),

                "embedding_size":
                    768,

                "hidden_size":
                    256,

                "best_validation_accuracy":
                    best_validation_accuracy
            },
            BEST_MODEL_PATH
        )

        saved_message = " ← BEST MODEL SAVED"

    else:

        saved_message = ""

    # ==============================================
    # Epoch results
    # ==============================================

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS}"
    )

    print(
        f"Train Loss: "
        f"{epoch_train_loss:.4f} | "
        f"Train Accuracy: "
        f"{epoch_train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss: "
        f"{epoch_validation_loss:.4f} | "
        f"Validation Accuracy: "
        f"{epoch_validation_accuracy * 100:.2f}%"
        f"{saved_message}"
    )

# --------------------------------------------------
# Final result
# --------------------------------------------------

print("\n" + "=" * 55)
print("COMPATIBILITY TRAINING COMPLETE")
print("=" * 55)

print(
    f"Best validation accuracy: "
    f"{best_validation_accuracy * 100:.2f}%"
)

print(
    "\nBest model saved to:"
)

print(BEST_MODEL_PATH)

Epoch 1/15:   0%|          | 0/425 [00:00<?, ?it/s]


Epoch 1/15
Train Loss: 0.6582 | Train Accuracy: 59.21%
Validation Loss: 0.6154 | Validation Accuracy: 66.46% ← BEST MODEL SAVED


Epoch 2/15:   0%|          | 0/425 [00:00<?, ?it/s]


Epoch 2/15
Train Loss: 0.6052 | Train Accuracy: 67.20%
Validation Loss: 0.5984 | Validation Accuracy: 68.23% ← BEST MODEL SAVED


Epoch 3/15:   0%|          | 0/425 [00:00<?, ?it/s]

...


Epoch 13/15
Train Loss: 0.2991 | Train Accuracy: 87.54%
Validation Loss: 0.7872 | Validation Accuracy: 69.59%


Epoch 14/15:   0%|          | 0/425 [00:00<?, ?it/s]


Epoch 14/15
Train Loss: 0.2801 | Train Accuracy: 88.65%
Validation Loss: 0.8361 | Validation Accuracy: 69.01%


Epoch 15/15:   0%|          | 0/425 [00:00<?, ?it/s]


Epoch 15/15
Train Loss: 0.2538 | Train Accuracy: 89.68%
Validation Loss: 0.8850 | Validation Accuracy: 68.03%

COMPATIBILITY TRAINING COMPLETE
Best validation accuracy: 71.79%

Best model saved to:
/content/drive/MyDrive/PsyStyle/models/compatibility/psystyle_compatibility.pt


In [ ]:
# ==================================================
# PSYSTYLE PSYCHOLOGY RULES
# ==================================================

# Each clothing category receives a psychological
# suitability score for different user contexts.

PSYCHOLOGY_RULES = {

    "Happy": {
        "Tees_Tanks": 0.90,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.75,
        "Sweaters": 0.70,
        "Shorts": 0.90,
        "Pants": 0.70,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.65,
        "Cardigans": 0.70,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.80,
        "Graphic_Tees": 0.95,
        "Denim": 0.85,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.70,
        "Leggings": 0.75
    },

    "Nervous": {
        "Tees_Tanks": 0.65,
        "Dresses": 0.60,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.85,
        "Shorts": 0.50,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.55,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.90,
        "Skirts": 0.60,
        "Sweatshirts_Hoodies": 0.85,
        "Graphic_Tees": 0.55,
        "Denim": 0.80,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.80,
        "Leggings": 0.75
    },

    "Confident": {
        "Tees_Tanks": 0.75,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.90,
        "Sweaters": 0.75,
        "Shorts": 0.75,
        "Pants": 0.90,
        "Rompers_Jumpsuits": 0.85,
        "Jackets_Coats": 0.95,
        "Cardigans": 0.70,
        "Skirts": 0.85,
        "Sweatshirts_Hoodies": 0.70,
        "Graphic_Tees": 0.80,
        "Denim": 0.85,
        "Shirts_Polos": 0.90,
        "Jackets_Vests": 0.90,
        "Leggings": 0.70
    }
}


# ==================================================
# PERSONALITY RULES
# ==================================================

PERSONALITY_RULES = {

    "Introvert": {
        "Tees_Tanks": 0.75,
        "Dresses": 0.70,
        "Blouses_Shirts": 0.85,
        "Sweaters": 0.90,
        "Shorts": 0.65,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.65,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.95,
        "Skirts": 0.70,
        "Sweatshirts_Hoodies": 0.90,
        "Graphic_Tees": 0.55,
        "Denim": 0.80,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.75,
        "Leggings": 0.80
    },

    "Extrovert": {
        "Tees_Tanks": 0.85,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.70,
        "Shorts": 0.85,
        "Pants": 0.75,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.65,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.70,
        "Graphic_Tees": 0.95,
        "Denim": 0.90,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.90,
        "Leggings": 0.70
    }
}


# ==================================================
# OCCASION RULES
# ==================================================

OCCASION_RULES = {

    "Interview": {
        "Tees_Tanks": 0.35,
        "Dresses": 0.80,
        "Blouses_Shirts": 0.95,
        "Sweaters": 0.75,
        "Shorts": 0.10,
        "Pants": 0.95,
        "Rompers_Jumpsuits": 0.50,
        "Jackets_Coats": 0.90,
        "Cardigans": 0.80,
        "Skirts": 0.85,
        "Sweatshirts_Hoodies": 0.20,
        "Graphic_Tees": 0.10,
        "Denim": 0.50,
        "Shirts_Polos": 0.90,
        "Jackets_Vests": 0.85,
        "Leggings": 0.25
    },

    "Party": {
        "Tees_Tanks": 0.80,
        "Dresses": 0.95,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.65,
        "Shorts": 0.85,
        "Pants": 0.75,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.80,
        "Cardigans": 0.55,
        "Skirts": 0.95,
        "Sweatshirts_Hoodies": 0.55,
        "Graphic_Tees": 0.90,
        "Denim": 0.85,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.90,
        "Leggings": 0.65
    },

    "College": {
        "Tees_Tanks": 0.90,
        "Dresses": 0.70,
        "Blouses_Shirts": 0.75,
        "Sweaters": 0.80,
        "Shorts": 0.85,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.75,
        "Jackets_Coats": 0.75,
        "Cardigans": 0.70,
        "Skirts": 0.70,
        "Sweatshirts_Hoodies": 0.90,
        "Graphic_Tees": 0.90,
        "Denim": 0.95,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.80,
        "Leggings": 0.75
    },

    "Wedding": {
        "Tees_Tanks": 0.20,
        "Dresses": 0.95,
        "Blouses_Shirts": 0.85,
        "Sweaters": 0.45,
        "Shorts": 0.05,
        "Pants": 0.70,
        "Rompers_Jumpsuits": 0.80,
        "Jackets_Coats": 0.70,
        "Cardigans": 0.45,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.05,
        "Graphic_Tees": 0.05,
        "Denim": 0.15,
        "Shirts_Polos": 0.60,
        "Jackets_Vests": 0.65,
        "Leggings": 0.10
    }
}


print("=" * 55)
print("PSYCHOLOGY RULES CREATED")
print("=" * 55)

print("Mood options:", list(PSYCHOLOGY_RULES.keys()))
print("Personality options:", list(PERSONALITY_RULES.keys()))
print("Occasion options:", list(OCCASION_RULES.keys()))

print(
    "\nClothing categories:",
    len(CLASS_NAMES)
)

PSYCHOLOGY RULES CREATED
Mood options: ['Happy', 'Nervous', 'Confident']
Personality options: ['Introvert', 'Extrovert']
Occasion options: ['Interview', 'Party', 'College', 'Wedding']

Clothing categories: 16


In [ ]:
def calculate_psychology_score(
    category,
    mood,
    personality,
    occasion,
    confidence
):
    """
    Calculate a psychology-based suitability score
    for one clothing category.

    Returns:
        score between 0 and 1
    """

    # ----------------------------------------------
    # Get rule scores
    # ----------------------------------------------

    mood_score = PSYCHOLOGY_RULES[
        mood
    ][category]

    personality_score = PERSONALITY_RULES[
        personality
    ][category]

    occasion_score = OCCASION_RULES[
        occasion
    ][category]

    # ----------------------------------------------
    # Normalize confidence
    # ----------------------------------------------

    confidence_normalized = (
        confidence / 10.0
    )

    # ----------------------------------------------
    # Combine psychological factors
    # ----------------------------------------------

    base_score = (
        0.40 * mood_score
        +
        0.30 * personality_score
        +
        0.30 * occasion_score
    )

    # ----------------------------------------------
    # Confidence adjustment
    #
    # Confidence affects how strongly the user
    # aligns with the psychological suitability.
    # ----------------------------------------------

    confidence_adjustment = (
        0.10
        *
        confidence_normalized
        *
        (mood_score - 0.5)
    )

    final_score = (
        base_score
        +
        confidence_adjustment
    )

    # ----------------------------------------------
    # Keep score between 0 and 1
    # ----------------------------------------------

    final_score = max(
        0.0,
        min(1.0, final_score)
    )

    return final_score


# ==================================================
# Test the psychology scoring
# ==================================================

test_category = "Blouses_Shirts"

test_score = calculate_psychology_score(
    category=test_category,
    mood="Confident",
    personality="Extrovert",
    occasion="Interview",
    confidence=8
)

print("=" * 55)
print("PSYCHOLOGY SCORING TEST")
print("=" * 55)

print("Category:", test_category)
print("Mood:", "Confident")
print("Personality:", "Extrovert")
print("Occasion:", "Interview")
print("Confidence:", "8/10")

print(
    "\nPsychology Score:",
    round(test_score, 4)
)

print(
    "Psychology Score (%):",
    f"{test_score * 100:.2f}%"
)

PSYCHOLOGY SCORING TEST
Category: Blouses_Shirts
Mood: Confident
Personality: Extrovert
Occasion: Interview
Confidence: 8/10

Psychology Score: 0.917
Psychology Score (%): 91.70%


In [ ]:
from PIL import Image
import torch


def predict_clothing_item(image):
    """
    Predict the clothing category and generate
    the 768-D ViT embedding for a wardrobe image.

    Returns:
        category
        confidence
        embedding
    """

    # ----------------------------------------------
    # Prepare image
    # ----------------------------------------------

    image = image.convert("RGB")

    processed = processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = (
        processed["pixel_values"]
        .to(DEVICE)
    )

    # ----------------------------------------------
    # ViT inference
    # ----------------------------------------------

    vit_classifier.eval()

    with torch.no_grad():

        logits, embedding = vit_classifier(
            pixel_values,
            return_embedding=True
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predicted_index = torch.argmax(
            probabilities,
            dim=1
        ).item()

        confidence = probabilities[
            0,
            predicted_index
        ].item()

    # ----------------------------------------------
    # Convert index → category
    # ----------------------------------------------

    category = CLASS_NAMES[
        predicted_index
    ]

    # ----------------------------------------------
    # Convert embedding to NumPy
    # ----------------------------------------------

    embedding = (
        embedding
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return (
        category,
        confidence,
        embedding
    )


print("=" * 55)
print("WARDROBE IMAGE PREDICTOR CREATED")
print("=" * 55)

print("Input: clothing image")
print("Output:")
print("  1. Clothing category")
print("  2. Prediction confidence")
print("  3. 768-D ViT embedding")

WARDROBE IMAGE PREDICTOR CREATED
Input: clothing image
Output:
  1. Clothing category
  2. Prediction confidence
  3. 768-D ViT embedding


In [ ]:
# --------------------------------------------------
# Test ViT wardrobe prediction
# --------------------------------------------------

import os
from PIL import Image

# Pick one Polyvore clothing image
test_filename = image_files[0]

test_image_path = os.path.join(
    POLYVORE_IMAGE_DIR,
    test_filename
)

test_image = Image.open(
    test_image_path
).convert("RGB")

# Run prediction
category, confidence, embedding = (
    predict_clothing_item(test_image)
)

# --------------------------------------------------
# Display result
# --------------------------------------------------

print("=" * 55)
print("WARDROBE PREDICTION TEST")
print("=" * 55)

print("Image:", test_filename)

print(
    "\nPredicted category:",
    category
)

print(
    "Prediction confidence:",
    f"{confidence * 100:.2f}%"
)

print(
    "Embedding shape:",
    embedding.shape
)

print(
    "\nEmbedding size:",
    embedding.shape[0]
)

WARDROBE PREDICTION TEST
Image: 100015595.jpg

Predicted category: Denim
Prediction confidence: 11.96%
Embedding shape: (768,)

Embedding size: 768


In [ ]:
import torch
import numpy as np

def calculate_compatibility_score(item_embeddings):
    """
    Calculate compatibility score for one outfit.

    Parameters
    ----------
    item_embeddings:
        List of 768-D ViT embeddings.

    Returns
    -------
    float:
        Compatibility score between 0 and 100.
    """

    # Convert the complete list to one NumPy array first
    embeddings = np.asarray(
        item_embeddings,
        dtype=np.float32
    )

    # Convert NumPy array to PyTorch tensor
    embeddings = torch.from_numpy(
        embeddings
    )

    # Add batch dimension
    embeddings = embeddings.unsqueeze(0)

    num_items = embeddings.shape[1]

    # Create mask
    mask = torch.ones(
        1,
        num_items,
        dtype=torch.bool
    )

    # Move to GPU
    embeddings = embeddings.to(DEVICE)
    mask = mask.to(DEVICE)

    # Compatibility prediction
    compatibility_model.eval()

    with torch.no_grad():

        logits = compatibility_model(
            embeddings,
            mask
        )

        probability = torch.sigmoid(
            logits
        ).item()

    return probability * 100


print("=" * 55)
print("COMPATIBILITY SCORING FUNCTION UPDATED")
print("=" * 55)

print("Input: list of 768-D clothing embeddings")
print("Output: compatibility score (0–100)")
print("Optimized NumPy → PyTorch conversion")

COMPATIBILITY SCORING FUNCTION UPDATED
Input: list of 768-D clothing embeddings
Output: compatibility score (0–100)
Optimized NumPy → PyTorch conversion


In [ ]:
import torch

# --------------------------------------------------
# Best Compatibility Network checkpoint
# --------------------------------------------------

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "PsyStyle/models/compatibility/"
    "psystyle_compatibility.pt"
)

# --------------------------------------------------
# Load checkpoint
# --------------------------------------------------

compatibility_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

# --------------------------------------------------
# Restore best model weights
# --------------------------------------------------

compatibility_model.load_state_dict(
    compatibility_checkpoint[
        "model_state_dict"
    ]
)

compatibility_model = compatibility_model.to(
    DEVICE
)

compatibility_model.eval()

# --------------------------------------------------
# Verify
# --------------------------------------------------

best_accuracy = compatibility_checkpoint[
    "best_validation_accuracy"
]

print("=" * 55)
print("BEST COMPATIBILITY MODEL LOADED")
print("=" * 55)

print(
    "Best validation accuracy:",
    f"{best_accuracy * 100:.2f}%"
)

print(
    "Embedding size:",
    compatibility_checkpoint[
        "embedding_size"
    ]
)

print(
    "Hidden size:",
    compatibility_checkpoint[
        "hidden_size"
    ]
)

print(
    "Device:",
    DEVICE
)

BEST COMPATIBILITY MODEL LOADED
Best validation accuracy: 71.79%
Embedding size: 768
Hidden size: 256
Device: cuda


In [ ]:
# --------------------------------------------------
# Test Compatibility Network
# --------------------------------------------------

test_sample = mapped_samples[0]

test_item_ids = test_sample["item_ids"]
test_label = test_sample["label"]

# Get embeddings for all items in the outfit
test_embeddings = [
    item_to_embedding[item_id]
    for item_id in test_item_ids
]

# Calculate compatibility
compatibility_score = (
    calculate_compatibility_score(
        test_embeddings
    )
)

print("=" * 55)
print("COMPATIBILITY NETWORK TEST")
print("=" * 55)

print(
    "Number of clothing items:",
    len(test_item_ids)
)

print(
    "Item IDs:",
    test_item_ids
)

print(
    "\nActual Polyvore label:",
    test_label
)

print(
    "Actual meaning:",
    "Compatible"
    if test_label == 1
    else "Incompatible"
)

print(
    "\nPredicted compatibility score:",
    f"{compatibility_score:.2f}%"
)

print(
    "Predicted class:",
    "Compatible"
    if compatibility_score >= 50
    else "Incompatible"
)

COMPATIBILITY NETWORK TEST
Number of clothing items: 6
Item IDs: ['132621870', '153967122', '171169800', '162799044', '172538912', '172312529']

Actual Polyvore label: 1
Actual meaning: Compatible

Predicted compatibility score: 66.32%
Predicted class: Compatible


/tmp/ipykernel_448/557799858.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  embeddings = torch.tensor(


In [ ]:
def calculate_outfit_psychology_score(
    categories,
    mood,
    personality,
    occasion,
    confidence
):
    """
    Calculate the psychology score for a complete outfit.

    Parameters
    ----------
    categories:
        List of clothing categories in the outfit.

    mood:
        User's current mood.

    personality:
        User's personality.

    occasion:
        Intended occasion.

    confidence:
        Confidence level from 1 to 10.

    Returns
    -------
    float:
        Outfit psychology score between 0 and 100.
    """

    item_scores = []

    for category in categories:

        score = calculate_psychology_score(
            category=category,
            mood=mood,
            personality=personality,
            occasion=occasion,
            confidence=confidence
        )

        item_scores.append(score)

    # Average psychology compatibility
    outfit_score = sum(item_scores) / len(item_scores)

    return outfit_score * 100


print("=" * 55)
print("OUTFIT PSYCHOLOGY SCORING FUNCTION CREATED")
print("=" * 55)

print("Input:")
print("  • Clothing categories")
print("  • Mood")
print("  • Personality")
print("  • Occasion")
print("  • Confidence")

print("\nOutput:")
print("  • Outfit psychology score (0–100)")

OUTFIT PSYCHOLOGY SCORING FUNCTION CREATED
Input:
  • Clothing categories
  • Mood
  • Personality
  • Occasion
  • Confidence

Output:
  • Outfit psychology score (0–100)


In [ ]:
# --------------------------------------------------
# Test Outfit Psychology Score
# --------------------------------------------------

test_categories = []

for item_id in test_item_ids:

    # Get the Polyvore image
    image_path = os.path.join(
        POLYVORE_IMAGE_DIR,
        f"{item_id}.jpg"
    )

    # Open image
    image = Image.open(
        image_path
    ).convert("RGB")

    # Predict clothing category
    category, prediction_confidence, _ = (
        predict_clothing_item(image)
    )

    test_categories.append(category)

    print(
        f"{item_id} → "
        f"{category} "
        f"({prediction_confidence * 100:.2f}%)"
    )


# --------------------------------------------------
# User psychological context
# --------------------------------------------------

test_mood = "Confident"
test_personality = "Extrovert"
test_occasion = "Interview"
test_confidence = 8


# --------------------------------------------------
# Calculate outfit psychology score
# --------------------------------------------------

psychology_score = (
    calculate_outfit_psychology_score(
        categories=test_categories,
        mood=test_mood,
        personality=test_personality,
        occasion=test_occasion,
        confidence=test_confidence
    )
)


print("\n" + "=" * 55)
print("OUTFIT PSYCHOLOGY TEST")
print("=" * 55)

print("Mood:", test_mood)
print("Personality:", test_personality)
print("Occasion:", test_occasion)
print("Confidence:", f"{test_confidence}/10")

print("\nPredicted categories:")
print(test_categories)

print(
    "\nPsychology Score:",
    f"{psychology_score:.2f}%"
)

132621870 → Jackets_Coats (18.04%)
153967122 → Dresses (23.57%)
171169800 → Shorts (13.27%)
162799044 → Sweaters (27.39%)
172538912 → Rompers_Jumpsuits (17.21%)
172312529 → Dresses (78.11%)

OUTFIT PSYCHOLOGY TEST
Mood: Confident
Personality: Extrovert
Occasion: Interview
Confidence: 8/10

Predicted categories:
['Jackets_Coats', 'Dresses', 'Shorts', 'Sweaters', 'Rompers_Jumpsuits', 'Dresses']

Psychology Score: 81.55%


In [ ]:
import torch
import torch.nn as nn


class PreferenceEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        preference_dim=256
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(
                embedding_dim,
                512
            ),
            nn.ReLU(),

            nn.Linear(
                512,
                preference_dim
            ),
            nn.ReLU()
        )

    def forward(
        self,
        outfit_embedding
    ):
        return self.encoder(
            outfit_embedding
        )


# --------------------------------------------------
# Create Preference Encoder
# --------------------------------------------------

preference_encoder = PreferenceEncoder(
    embedding_dim=768,
    preference_dim=256
)

preference_encoder = preference_encoder.to(
    DEVICE
)

preference_encoder.eval()


print("=" * 55)
print("PREFERENCE ENCODER CREATED")
print("=" * 55)

print("Input: 768-D outfit representation")
print("Output: 256-D preference vector")

print(
    "Device:",
    DEVICE
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in preference_encoder.parameters()
    )
)

PREFERENCE ENCODER CREATED
Input: 768-D outfit representation
Output: 256-D preference vector
Device: cuda
Trainable parameters: 525056


In [ ]:
import torch
import torch.nn.functional as F


def create_outfit_representation(
    item_embeddings
):
    """
    Convert multiple item embeddings into
    one outfit representation.
    """

    embeddings = torch.tensor(
        item_embeddings,
        dtype=torch.float32,
        device=DEVICE
    )

    # Average the clothing-item embeddings
    outfit_embedding = embeddings.mean(
        dim=0
    )

    return outfit_embedding


def encode_outfit_preference(
    item_embeddings
):
    """
    Convert an outfit into a 256-D
    preference representation.
    """

    outfit_embedding = (
        create_outfit_representation(
            item_embeddings
        )
    )

    outfit_embedding = outfit_embedding.unsqueeze(0)

    preference_vector = (
        preference_encoder(
            outfit_embedding
        )
    )

    preference_vector = F.normalize(
        preference_vector,
        p=2,
        dim=1
    )

    return preference_vector.squeeze(0)


def update_preference_vector(
    current_preference,
    outfit_preference,
    feedback,
    learning_rate=0.20
):
    """
    Update the user's preference vector.

    feedback:
        "like"    → positive update
        "dislike" → negative update
        "skip"    → no update
    """

    if feedback == "like":

        updated_preference = (
            current_preference
            +
            learning_rate
            * outfit_preference
        )

    elif feedback == "dislike":

        updated_preference = (
            current_preference
            -
            learning_rate
            * outfit_preference
        )

    elif feedback == "skip":

        updated_preference = (
            current_preference
        )

    else:
        raise ValueError(
            "Feedback must be: "
            "'like', 'dislike', or 'skip'"
        )

    # Normalize the updated vector
    updated_preference = F.normalize(
        updated_preference,
        p=2,
        dim=0
    )

    return updated_preference


print("=" * 55)
print("PREFERENCE UPDATE SYSTEM CREATED")
print("=" * 55)

print("❤️ Like    → positive preference update")
print("👎 Dislike → negative preference update")
print("— Skip     → no preference update")

print("\nPreference vector size: 256")

PREFERENCE UPDATE SYSTEM CREATED
❤️ Like    → positive preference update
👎 Dislike → negative preference update
— Skip     → no preference update

Preference vector size: 256


In [ ]:
# --------------------------------------------------
# Initialize user's preference vector
# --------------------------------------------------

PREFERENCE_DIM = 256

user_preference_vector = torch.zeros(
    PREFERENCE_DIM,
    dtype=torch.float32,
    device=DEVICE
)

print("=" * 55)
print("USER PREFERENCE VECTOR INITIALIZED")
print("=" * 55)

print(
    "Preference vector size:",
    user_preference_vector.shape
)

print(
    "Initial vector norm:",
    user_preference_vector.norm().item()
)

print(
    "\nInitial state:",
    "No user feedback yet"
)

USER PREFERENCE VECTOR INITIALIZED
Preference vector size: torch.Size([256])
Initial vector norm: 0.0

Initial state: No user feedback yet


In [ ]:
# --------------------------------------------------
# Test Preference Learning
# --------------------------------------------------

# Use the same Polyvore outfit from our previous tests
preference_test_embeddings = [
    item_to_embedding[item_id]
    for item_id in test_item_ids
]

# Encode the outfit into a 256-D preference representation
outfit_preference = encode_outfit_preference(
    preference_test_embeddings
)

print("=" * 55)
print("PREFERENCE ENCODER TEST")
print("=" * 55)

print(
    "Outfit preference vector shape:",
    outfit_preference.shape
)

print(
    "Outfit preference vector norm:",
    f"{outfit_preference.norm().item():.4f}"
)


# --------------------------------------------------
# Test LIKE
# --------------------------------------------------

like_vector = update_preference_vector(
    current_preference=user_preference_vector,
    outfit_preference=outfit_preference,
    feedback="like"
)


# --------------------------------------------------
# Test DISLIKE
# --------------------------------------------------

dislike_vector = update_preference_vector(
    current_preference=user_preference_vector,
    outfit_preference=outfit_preference,
    feedback="dislike"
)


# --------------------------------------------------
# Test SKIP
# --------------------------------------------------

skip_vector = update_preference_vector(
    current_preference=user_preference_vector,
    outfit_preference=outfit_preference,
    feedback="skip"
)


# --------------------------------------------------
# Compare results
# --------------------------------------------------

print("\n" + "=" * 55)
print("FEEDBACK UPDATE TEST")
print("=" * 55)

print(
    "Initial preference norm:",
    f"{user_preference_vector.norm().item():.4f}"
)

print(
    "After LIKE norm:",
    f"{like_vector.norm().item():.4f}"
)

print(
    "After DISLIKE norm:",
    f"{dislike_vector.norm().item():.4f}"
)

print(
    "After SKIP norm:",
    f"{skip_vector.norm().item():.4f}"
)

print(
    "\nLIKE and DISLIKE produce different preference states."
)

print(
    "SKIP keeps the preference vector unchanged."
)

PREFERENCE ENCODER TEST
Outfit preference vector shape: torch.Size([256])
Outfit preference vector norm: 1.0000

FEEDBACK UPDATE TEST
Initial preference norm: 0.0000
After LIKE norm: 1.0000
After DISLIKE norm: 1.0000
After SKIP norm: 0.0000

LIKE and DISLIKE produce different preference states.
SKIP keeps the preference vector unchanged.


In [ ]:
import torch
import torch.nn as nn


class FusionNetwork(nn.Module):

    def __init__(
        self,
        preference_dim=256
    ):
        super().__init__()

        # Input:
        # 1 compatibility score
        # 1 psychology score
        # 256-D preference vector

        input_dim = 258

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            ),

            nn.Sigmoid()
        )

    def forward(
        self,
        compatibility_score,
        psychology_score,
        preference_vector
    ):

        # Convert scores from 0–100
        # into 0–1

        compatibility_score = (
            compatibility_score / 100.0
        )

        psychology_score = (
            psychology_score / 100.0
        )

        # Add batch dimensions
        compatibility_score = (
            compatibility_score.view(-1, 1)
        )

        psychology_score = (
            psychology_score.view(-1, 1)
        )

        preference_vector = (
            preference_vector.view(
                -1,
                preference_vector.shape[-1]
            )
        )

        # Combine all three inputs
        combined = torch.cat(
            [
                compatibility_score,
                psychology_score,
                preference_vector
            ],
            dim=1
        )

        # Final score
        final_score = self.network(
            combined
        )

        return final_score.squeeze(-1)


# --------------------------------------------------
# Create Fusion Network
# --------------------------------------------------

fusion_model = FusionNetwork(
    preference_dim=256
)

fusion_model = fusion_model.to(
    DEVICE
)


print("=" * 55)
print("FUSION NETWORK CREATED")
print("=" * 55)

print("Compatibility input: 1")
print("Psychology input: 1")
print("Preference input: 256")

print("Total input size: 258")
print("Output: Final outfit score (0–1)")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in fusion_model.parameters()
    )
)

print(
    "Device:",
    DEVICE
)

FUSION NETWORK CREATED
Compatibility input: 1
Psychology input: 1
Preference input: 256
Total input size: 258
Output: Final outfit score (0–1)
Trainable parameters: 41473
Device: cuda


In [ ]:
import numpy as np
import torch


# --------------------------------------------------
# Fusion training configuration
# --------------------------------------------------

FUSION_COMPATIBILITY_WEIGHT = 0.60
FUSION_PSYCHOLOGY_WEIGHT = 0.40

FUSION_MOOD = "Confident"
FUSION_PERSONALITY = "Extrovert"
FUSION_OCCASION = "Interview"
FUSION_CONFIDENCE = 8


print("=" * 55)
print("FUSION TRAINING CONFIGURATION")
print("=" * 55)

print(
    "Compatibility weight:",
    FUSION_COMPATIBILITY_WEIGHT
)

print(
    "Psychology weight:",
    FUSION_PSYCHOLOGY_WEIGHT
)

print(
    "\nPsychological context:"
)

print("Mood:", FUSION_MOOD)
print("Personality:", FUSION_PERSONALITY)
print("Occasion:", FUSION_OCCASION)
print("Confidence:", f"{FUSION_CONFIDENCE}/10")

print(
    "\nTarget formula:"
)

print(
    "Target = "
    "0.60 × Compatibility + "
    "0.40 × Psychology"
)

FUSION TRAINING CONFIGURATION
Compatibility weight: 0.6
Psychology weight: 0.4

Psychological context:
Mood: Confident
Personality: Extrovert
Occasion: Interview
Confidence: 8/10

Target formula:
Target = 0.60 × Compatibility + 0.40 × Psychology


In [ ]:
# --------------------------------------------------
# Prepare Polyvore category mapping for Fusion
# --------------------------------------------------

POLYVORE_TO_PSYSTYLE = {
    "tops": "Blouses_Shirts",
    "bottoms": "Pants",
    "dresses": "Dresses",
    "outerwear": "Jackets_Coats",
    "skirts": "Skirts",
    "shorts": "Shorts",
    "jumpsuits": "Rompers_Jumpsuits",
    "sweaters": "Sweaters",
    "cardigans": "Cardigans",
    "hoodies": "Sweatshirts_Hoodies",
    "jeans": "Denim",
    "leggings": "Leggings"
}

print("=" * 55)
print("POLYVORE → PSYSTYLE CATEGORY MAPPING")
print("=" * 55)

print(
    "Mapped Polyvore categories:",
    len(POLYVORE_TO_PSYSTYLE)
)

for polyvore_category, psystyle_category in (
    POLYVORE_TO_PSYSTYLE.items()
):
    print(
        f"{polyvore_category:15} → "
        f"{psystyle_category}"
    )

POLYVORE → PSYSTYLE CATEGORY MAPPING
Mapped Polyvore categories: 12
tops            → Blouses_Shirts
bottoms         → Pants
dresses         → Dresses
outerwear       → Jackets_Coats
skirts          → Skirts
shorts          → Shorts
jumpsuits       → Rompers_Jumpsuits
sweaters        → Sweaters
cardigans       → Cardigans
hoodies         → Sweatshirts_Hoodies
jeans           → Denim
leggings        → Leggings


In [ ]:
import json
from collections import Counter

# --------------------------------------------------
# Load Polyvore metadata
# --------------------------------------------------

POLYVORE_METADATA_PATH = (
    "/content/psystyle_data/"
    "polyvore_hf/"
    "polyvore_item_metadata.json"
)

with open(
    POLYVORE_METADATA_PATH,
    "r"
) as f:
    polyvore_metadata = json.load(f)


# --------------------------------------------------
# Count semantic categories
# --------------------------------------------------

semantic_category_counts = Counter()

for item_id, metadata in polyvore_metadata.items():

    category = metadata.get(
        "semantic_category",
        ""
    )

    if category:
        semantic_category_counts[
            category
        ] += 1


# --------------------------------------------------
# Display categories
# --------------------------------------------------

print("=" * 55)
print("POLYVORE SEMANTIC CATEGORIES")
print("=" * 55)

print(
    "Total metadata items:",
    len(polyvore_metadata)
)

print(
    "Categories found:",
    len(semantic_category_counts)
)

print("\nCategory counts:\n")

for category, count in (
    semantic_category_counts.most_common()
):
    print(
        f"{category:25} {count:,}"
    )

POLYVORE SEMANTIC CATEGORIES
Total metadata items: 251008
Categories found: 11

Category counts:

shoes                     44,850
jewellery                 41,414
bags                      40,717
tops                      32,998
bottoms                   27,670
all-body                  18,478
outerwear                 17,065
sunglasses                9,874
accessories               6,973
hats                      6,071
scarves                   4,898


In [ ]:
# --------------------------------------------------
# Check Polyvore compatibility coverage
# --------------------------------------------------

mapped_item_count = 0
unmapped_item_count = 0

mapped_categories = Counter()
unmapped_categories = Counter()

for item_id in item_to_embedding.keys():

    metadata = polyvore_metadata.get(
        str(item_id)
    )

    if metadata is None:
        unmapped_item_count += 1
        continue

    semantic_category = metadata.get(
        "semantic_category",
        ""
    )

    if semantic_category in POLYVORE_TO_PSYSTYLE:

        mapped_item_count += 1

        mapped_categories[
            semantic_category
        ] += 1

    else:

        unmapped_item_count += 1

        unmapped_categories[
            semantic_category
        ] += 1


print("=" * 55)
print("POLYVORE CATEGORY COVERAGE")
print("=" * 55)

print(
    "Total embedded items:",
    len(item_to_embedding)
)

print(
    "Mapped items:",
    mapped_item_count
)

print(
    "Unmapped items:",
    unmapped_item_count
)

print(
    "Mapped percentage:",
    f"{mapped_item_count / len(item_to_embedding) * 100:.2f}%"
)

print("\nMapped categories:")

for category, count in (
    mapped_categories.most_common()
):
    print(
        f"  {category:15} {count:,}"
    )

print("\nUnmapped categories:")

for category, count in (
    unmapped_categories.most_common()
):
    print(
        f"  {category:15} {count:,}"
    )

POLYVORE CATEGORY COVERAGE
Total embedded items: 71967
Mapped items: 21175
Unmapped items: 50792
Mapped percentage: 29.42%

Mapped categories:
  tops            9,136
  bottoms         7,886
  outerwear       4,153

Unmapped categories:
  shoes           13,362
  jewellery       12,067
  bags            11,832
  all-body        5,751
  sunglasses      3,294
  accessories     1,692
  hats            1,676
  scarves         1,118


In [ ]:
# --------------------------------------------------
# Build valid Fusion samples
# --------------------------------------------------

valid_fusion_samples = []

skipped_samples = 0

for sample in mapped_samples:

    item_ids = sample["item_ids"]

    categories = []
    valid = True

    for item_id in item_ids:

        metadata = polyvore_metadata.get(
            str(item_id)
        )

        if metadata is None:
            valid = False
            break

        semantic_category = metadata.get(
            "semantic_category",
            ""
        )

        # Only use categories that we can
        # reliably map to PsyStyle categories
        if semantic_category not in (
            POLYVORE_TO_PSYSTYLE
        ):
            valid = False
            break

        psystyle_category = (
            POLYVORE_TO_PSYSTYLE[
                semantic_category
            ]
        )

        categories.append(
            psystyle_category
        )

    if valid:

        valid_fusion_samples.append({
            "item_ids": item_ids,
            "categories": categories,
            "label": sample["label"]
        })

    else:
        skipped_samples += 1


# --------------------------------------------------
# Statistics
# --------------------------------------------------

print("=" * 55)
print("VALID FUSION SAMPLES CREATED")
print("=" * 55)

print(
    "Original compatibility samples:",
    len(mapped_samples)
)

print(
    "Valid Fusion samples:",
    len(valid_fusion_samples)
)

print(
    "Skipped samples:",
    skipped_samples
)

if len(valid_fusion_samples) > 0:

    labels = [
        sample["label"]
        for sample in valid_fusion_samples
    ]

    compatible_count = sum(
        label == 1
        for label in labels
    )

    incompatible_count = sum(
        label == 0
        for label in labels
    )

    print(
        "\nCompatible samples:",
        compatible_count
    )

    print(
        "Incompatible samples:",
        incompatible_count
    )

    print(
        "\nExample categories:"
    )

    print(
        valid_fusion_samples[0][
            "categories"
        ]
    )

VALID FUSION SAMPLES CREATED
Original compatibility samples: 33990
Valid Fusion samples: 76
Skipped samples: 33914

Compatible samples: 38
Incompatible samples: 38

Example categories:
['Blouses_Shirts', 'Pants', 'Jackets_Coats']


In [ ]:
import torch
import torch.nn.functional as F


def calculate_preference_score(
    user_preference,
    outfit_preference
):
    """
    Calculate how well an outfit matches
    the user's learned preferences.

    Returns a score between 0 and 100.

    New users with no feedback receive
    a neutral score of 50.
    """

    # --------------------------------------------------
    # No feedback yet
    # --------------------------------------------------

    if torch.norm(user_preference) < 1e-8:

        return 50.0


    # --------------------------------------------------
    # Normalize both vectors
    # --------------------------------------------------

    user_vector = F.normalize(
        user_preference.unsqueeze(0),
        p=2,
        dim=1
    )

    outfit_vector = F.normalize(
        outfit_preference.unsqueeze(0),
        p=2,
        dim=1
    )


    # --------------------------------------------------
    # Cosine similarity
    # --------------------------------------------------

    similarity = F.cosine_similarity(
        user_vector,
        outfit_vector,
        dim=1
    ).item()


    # --------------------------------------------------
    # Convert [-1, 1] → [0, 100]
    # --------------------------------------------------

    score = (
        (similarity + 1.0)
        / 2.0
        * 100.0
    )

    return score


print("=" * 55)
print("PREFERENCE SCORING FUNCTION CREATED")
print("=" * 55)

print("Input:")
print("  • User preference vector")
print("  • Outfit preference vector")

print("\nMethod:")
print("  • Cosine similarity")

print("\nOutput:")
print("  • Preference score (0–100)")

print("\nNew user:")
print("  • Neutral preference score = 50")

PREFERENCE SCORING FUNCTION CREATED
Input:
  • User preference vector
  • Outfit preference vector

Method:
  • Cosine similarity

Output:
  • Preference score (0–100)

New user:
  • Neutral preference score = 50


In [ ]:
def calculate_final_outfit_score(
    compatibility_score,
    psychology_score,
    preference_score
):
    """
    Calculate the final PsyStyle outfit score.

    Scores are expected in the range 0–100.
    """

    # --------------------------------------------------
    # Fusion weights
    # --------------------------------------------------

    compatibility_weight = 0.50
    psychology_weight = 0.30
    preference_weight = 0.20

    # --------------------------------------------------
    # Final score
    # --------------------------------------------------

    final_score = (
        compatibility_weight
        * compatibility_score

        +

        psychology_weight
        * psychology_score

        +

        preference_weight
        * preference_score
    )

    return final_score


print("=" * 55)
print("FINAL OUTFIT SCORING FUNCTION CREATED")
print("=" * 55)

print("Compatibility weight: 50%")
print("Psychology weight:    30%")
print("Preference weight:    20%")

print("\nOutput:")
print("  Final Outfit Score (0–100)")

FINAL OUTFIT SCORING FUNCTION CREATED
Compatibility weight: 50%
Psychology weight:    30%
Preference weight:    20%

Output:
  Final Outfit Score (0–100)


In [ ]:
# --------------------------------------------------
# Complete PsyStyle scoring test
# --------------------------------------------------

# --------------------------------------------------
# 1. Compatibility score
# --------------------------------------------------

compatibility_score = (
    calculate_compatibility_score(
        test_embeddings
    )
)


# --------------------------------------------------
# 2. Psychology score
# --------------------------------------------------

psychology_score = (
    calculate_outfit_psychology_score(
        categories=test_categories,
        mood=test_mood,
        personality=test_personality,
        occasion=test_occasion,
        confidence=test_confidence
    )
)


# --------------------------------------------------
# 3. Outfit preference representation
# --------------------------------------------------

outfit_preference = (
    encode_outfit_preference(
        test_embeddings
    )
)


# --------------------------------------------------
# 4. Preference score
# --------------------------------------------------

preference_score = (
    calculate_preference_score(
        user_preference=user_preference_vector,
        outfit_preference=outfit_preference
    )
)


# --------------------------------------------------
# 5. Final PsyStyle score
# --------------------------------------------------

final_score = (
    calculate_final_outfit_score(
        compatibility_score=compatibility_score,
        psychology_score=psychology_score,
        preference_score=preference_score
    )
)


# --------------------------------------------------
# Display complete result
# --------------------------------------------------

print("=" * 55)
print("PSYSTYLE COMPLETE SCORING TEST")
print("=" * 55)

print("\nUser Context")
print("-" * 30)

print("Mood:", test_mood)
print("Personality:", test_personality)
print("Occasion:", test_occasion)
print("Confidence:", f"{test_confidence}/10")


print("\nOutfit")
print("-" * 30)

print("Items:", len(test_item_ids))
print("Categories:", test_categories)


print("\nScores")
print("-" * 30)

print(
    "Compatibility:",
    f"{compatibility_score:.2f}%"
)

print(
    "Psychology:",
    f"{psychology_score:.2f}%"
)

print(
    "Preference:",
    f"{preference_score:.2f}%"
)

print(
    "Final PsyStyle Score:",
    f"{final_score:.2f}%"
)

PSYSTYLE COMPLETE SCORING TEST

User Context
------------------------------
Mood: Confident
Personality: Extrovert
Occasion: Interview
Confidence: 8/10

Outfit
------------------------------
Items: 6
Categories: ['Jackets_Coats', 'Dresses', 'Shorts', 'Sweaters', 'Rompers_Jumpsuits', 'Dresses']

Scores
------------------------------
Compatibility: 66.32%
Psychology: 81.55%
Preference: 50.00%
Final PsyStyle Score: 67.63%


In [ ]:
from itertools import combinations


# --------------------------------------------------
# Outfit combination rules
# --------------------------------------------------

OUTFIT_RULES = {
    "top": [
        "Tees_Tanks",
        "Blouses_Shirts",
        "Sweaters",
        "Sweatshirts_Hoodies",
        "Graphic_Tees",
        "Shirts_Polos"
    ],

    "bottom": [
        "Pants",
        "Shorts",
        "Denim",
        "Skirts",
        "Leggings"
    ],

    "outerwear": [
        "Jackets_Coats",
        "Cardigans",
        "Jackets_Vests"
    ],

    "one_piece": [
        "Dresses",
        "Rompers_Jumpsuits"
    ]
}


def get_outfit_role(category):

    for role, categories in OUTFIT_RULES.items():

        if category in categories:
            return role

    return None


def generate_outfit_combinations(
    wardrobe_items
):
    """
    Generate valid outfit combinations.

    Each wardrobe item must contain:
        category
        embedding
        image
    """

    outfits = []

    tops = []
    bottoms = []
    outerwear = []
    one_pieces = []

    # --------------------------------------------------
    # Categorize wardrobe items
    # --------------------------------------------------

    for item in wardrobe_items:

        role = get_outfit_role(
            item["category"]
        )

        if role == "top":
            tops.append(item)

        elif role == "bottom":
            bottoms.append(item)

        elif role == "outerwear":
            outerwear.append(item)

        elif role == "one_piece":
            one_pieces.append(item)


    # --------------------------------------------------
    # Top + Bottom
    # --------------------------------------------------

    for top in tops:

        for bottom in bottoms:

            outfits.append([
                top,
                bottom
            ])


    # --------------------------------------------------
    # Top + Bottom + Outerwear
    # --------------------------------------------------

    for top in tops:

        for bottom in bottoms:

            for outer in outerwear:

                outfits.append([
                    top,
                    bottom,
                    outer
                ])


    # --------------------------------------------------
    # One-piece outfits
    # --------------------------------------------------

    for one_piece in one_pieces:

        outfits.append([
            one_piece
        ])

        for outer in outerwear:

            outfits.append([
                one_piece,
                outer
            ])


    return outfits


print("=" * 55)
print("OUTFIT COMBINATION GENERATOR CREATED")
print("=" * 55)

print("Supported combinations:")

print("  • Top + Bottom")
print("  • Top + Bottom + Outerwear")
print("  • One-piece")
print("  • One-piece + Outerwear")

OUTFIT COMBINATION GENERATOR CREATED
Supported combinations:
  • Top + Bottom
  • Top + Bottom + Outerwear
  • One-piece
  • One-piece + Outerwear


In [ ]:
def score_outfit(
    outfit,
    mood,
    personality,
    occasion,
    confidence,
    user_preference
):
    """
    Calculate all PsyStyle scores for one outfit.
    """

    # --------------------------------------------------
    # Extract embeddings and categories
    # --------------------------------------------------

    embeddings = [
        item["embedding"]
        for item in outfit
    ]

    categories = [
        item["category"]
        for item in outfit
    ]


    # --------------------------------------------------
    # 1. Compatibility Score
    # --------------------------------------------------

    compatibility_score = (
        calculate_compatibility_score(
            embeddings
        )
    )


    # --------------------------------------------------
    # 2. Psychology Score
    # --------------------------------------------------

    psychology_score = (
        calculate_outfit_psychology_score(
            categories=categories,
            mood=mood,
            personality=personality,
            occasion=occasion,
            confidence=confidence
        )
    )


    # --------------------------------------------------
    # 3. Preference Representation
    # --------------------------------------------------

    outfit_preference = (
        encode_outfit_preference(
            embeddings
        )
    )


    # --------------------------------------------------
    # 4. Preference Score
    # --------------------------------------------------

    preference_score = (
        calculate_preference_score(
            user_preference=user_preference,
            outfit_preference=outfit_preference
        )
    )


    # --------------------------------------------------
    # 5. Final Score
    # --------------------------------------------------

    final_score = (
        calculate_final_outfit_score(
            compatibility_score=compatibility_score,
            psychology_score=psychology_score,
            preference_score=preference_score
        )
    )


    # --------------------------------------------------
    # Return complete result
    # --------------------------------------------------

    return {
        "items": outfit,
        "categories": categories,
        "compatibility_score": compatibility_score,
        "psychology_score": psychology_score,
        "preference_score": preference_score,
        "final_score": final_score
    }


print("=" * 55)
print("OUTFIT SCORING PIPELINE CREATED")
print("=" * 55)

print("Each outfit now receives:")
print("  • Compatibility Score")
print("  • Psychology Score")
print("  • Preference Score")
print("  • Final PsyStyle Score")

OUTFIT SCORING PIPELINE CREATED
Each outfit now receives:
  • Compatibility Score
  • Psychology Score
  • Preference Score
  • Final PsyStyle Score


In [ ]:
# --------------------------------------------------
# Create a small test wardrobe
# --------------------------------------------------

test_wardrobe = [
    {
        "name": "Blue Shirt",
        "category": "Blouses_Shirts",
        "embedding": item_to_embedding[test_item_ids[0]]
    },
    {
        "name": "Black Pants",
        "category": "Pants",
        "embedding": item_to_embedding[test_item_ids[1]]
    },
    {
        "name": "Denim Jacket",
        "category": "Jackets_Coats",
        "embedding": item_to_embedding[test_item_ids[2]]
    }
]


# --------------------------------------------------
# Generate outfits
# --------------------------------------------------

generated_outfits = (
    generate_outfit_combinations(
        test_wardrobe
    )
)


print("=" * 55)
print("TEST OUTFIT GENERATION")
print("=" * 55)

print(
    "Generated outfits:",
    len(generated_outfits)
)


# --------------------------------------------------
# Score every generated outfit
# --------------------------------------------------

scored_outfits = []

for outfit in generated_outfits:

    result = score_outfit(
        outfit=outfit,
        mood="Confident",
        personality="Extrovert",
        occasion="Interview",
        confidence=8,
        user_preference=user_preference_vector
    )

    scored_outfits.append(result)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("\n" + "=" * 55)
print("OUTFIT SCORES")
print("=" * 55)

for index, result in enumerate(
    scored_outfits,
    start=1
):

    names = [
        item["name"]
        for item in result["items"]
    ]

    print(f"\nOutfit {index}")
    print("-" * 30)

    print(
        "Items:",
        " + ".join(names)
    )

    print(
        "Categories:",
        result["categories"]
    )

    print(
        "Compatibility:",
        f"{result['compatibility_score']:.2f}%"
    )

    print(
        "Psychology:",
        f"{result['psychology_score']:.2f}%"
    )

    print(
        "Preference:",
        f"{result['preference_score']:.2f}%"
    )

    print(
        "Final Score:",
        f"{result['final_score']:.2f}%"
    )

TEST OUTFIT GENERATION
Generated outfits: 2

OUTFIT SCORES

Outfit 1
------------------------------
Items: Blue Shirt + Black Pants
Categories: ['Blouses_Shirts', 'Pants']
Compatibility: 87.06%
Psychology: 90.95%
Preference: 50.00%
Final Score: 80.82%

Outfit 2
------------------------------
Items: Blue Shirt + Black Pants + Denim Jacket
Categories: ['Blouses_Shirts', 'Pants', 'Jackets_Coats']
Compatibility: 77.74%
Psychology: 92.00%
Preference: 50.00%
Final Score: 76.47%


In [ ]:
def rank_outfits(scored_outfits):
    """
    Sort outfits from highest to lowest
    according to the final PsyStyle score.
    """

    ranked_outfits = sorted(
        scored_outfits,
        key=lambda outfit: outfit["final_score"],
        reverse=True
    )

    return ranked_outfits


# --------------------------------------------------
# Rank the generated outfits
# --------------------------------------------------

ranked_outfits = rank_outfits(
    scored_outfits
)


# --------------------------------------------------
# Display ranking
# --------------------------------------------------

print("=" * 55)
print("PSYSTYLE OUTFIT RANKING")
print("=" * 55)

for rank, result in enumerate(
    ranked_outfits,
    start=1
):

    names = [
        item["name"]
        for item in result["items"]
    ]

    print(f"\nRank {rank}")
    print("-" * 30)

    print(
        "Outfit:",
        " + ".join(names)
    )

    print(
        "Final Score:",
        f"{result['final_score']:.2f}%"
    )

print("\n" + "=" * 55)
print("RANKING COMPLETE")
print("=" * 55)

PSYSTYLE OUTFIT RANKING

Rank 1
------------------------------
Outfit: Blue Shirt + Black Pants
Final Score: 80.82%

Rank 2
------------------------------
Outfit: Blue Shirt + Black Pants + Denim Jacket
Final Score: 76.47%

RANKING COMPLETE


In [ ]:
from PIL import Image


def process_wardrobe_image(
    image,
    name="Wardrobe Item"
):
    """
    Process one wardrobe image using
    the fine-tuned PsyStyle ViT.
    """

    # --------------------------------------------------
    # Ensure RGB image
    # --------------------------------------------------

    image = image.convert("RGB")


    # --------------------------------------------------
    # ViT prediction
    # --------------------------------------------------

    category, confidence, embedding = (
        predict_clothing_item(image)
    )


    # --------------------------------------------------
    # Create wardrobe item
    # --------------------------------------------------

    wardrobe_item = {
        "name": name,
        "image": image,
        "category": category,
        "confidence": confidence,
        "embedding": embedding
    }


    return wardrobe_item


print("=" * 55)
print("REAL WARDROBE IMAGE PROCESSOR CREATED")
print("=" * 55)

print("Input:")
print("  • Clothing image")

print("\nViT processing:")
print("  • Clothing category")
print("  • Prediction confidence")
print("  • 768-D embedding")

print("\nOutput:")
print("  • PsyStyle wardrobe item")

REAL WARDROBE IMAGE PROCESSOR CREATED
Input:
  • Clothing image

ViT processing:
  • Clothing category
  • Prediction confidence
  • 768-D embedding

Output:
  • PsyStyle wardrobe item


In [ ]:
import os
from PIL import Image


# --------------------------------------------------
# Select one real clothing image
# --------------------------------------------------

sample_filename = test_df.iloc[0]["filename"]

sample_path = os.path.join(
    TEST_DIR,
    sample_filename
)


# --------------------------------------------------
# Load image
# --------------------------------------------------

sample_image = Image.open(
    sample_path
).convert("RGB")


# --------------------------------------------------
# Process through PsyStyle ViT
# --------------------------------------------------

sample_wardrobe_item = (
    process_wardrobe_image(
        image=sample_image,
        name=sample_filename
    )
)


# --------------------------------------------------
# Display result
# --------------------------------------------------

print("=" * 55)
print("REAL WARDROBE IMAGE TEST")
print("=" * 55)

print("\nImage:")
print(sample_filename)

print(
    "Predicted category:",
    sample_wardrobe_item["category"]
)

print(
    "Confidence:",
    f"{sample_wardrobe_item['confidence'] * 100:.2f}%"
)

print(
    "Embedding shape:",
    sample_wardrobe_item["embedding"].shape
)

REAL WARDROBE IMAGE TEST

Image:
WOMEN-Blouses_Shirts-id_00000997-01_4_full.png
Predicted category: Blouses_Shirts
Confidence: 77.67%
Embedding shape: (768,)


In [ ]:
def process_wardrobe_images(
    image_paths
):
    """
    Process multiple wardrobe images
    using the fine-tuned PsyStyle ViT.
    """

    wardrobe_items = []

    for image_path in image_paths:

        # --------------------------------------------------
        # Load image
        # --------------------------------------------------

        image = Image.open(
            image_path
        ).convert("RGB")


        # --------------------------------------------------
        # Process image through ViT
        # --------------------------------------------------

        item = process_wardrobe_image(
            image=image,
            name=os.path.basename(image_path)
        )


        # --------------------------------------------------
        # Store result
        # --------------------------------------------------

        wardrobe_items.append(item)


    return wardrobe_items


print("=" * 55)
print("MULTI-IMAGE WARDROBE PROCESSOR CREATED")
print("=" * 55)

print("Input:")
print("  • Multiple clothing image paths")

print("\nOutput:")
print("  • Processed wardrobe items")
print("  • Category")
print("  • Confidence")
print("  • 768-D embedding")

MULTI-IMAGE WARDROBE PROCESSOR CREATED
Input:
  • Multiple clothing image paths

Output:
  • Processed wardrobe items
  • Category
  • Confidence
  • 768-D embedding


In [ ]:
# --------------------------------------------------
# Select 3 real clothing images
# --------------------------------------------------

sample_paths = [
    train_df.iloc[0]["filepath"],
    train_df.iloc[100]["filepath"],
    train_df.iloc[200]["filepath"]
]


# --------------------------------------------------
# Process all images
# --------------------------------------------------

test_wardrobe_items = (
    process_wardrobe_images(
        sample_paths
    )
)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("=" * 55)
print("MULTI-IMAGE WARDROBE TEST")
print("=" * 55)

print(
    "\nNumber of wardrobe items:",
    len(test_wardrobe_items)
)


for index, item in enumerate(
    test_wardrobe_items,
    start=1
):

    print(f"\nItem {index}")
    print("-" * 30)

    print(
        "Name:",
        item["name"]
    )

    print(
        "Category:",
        item["category"]
    )

    print(
        "Confidence:",
        f"{item['confidence'] * 100:.2f}%"
    )

    print(
        "Embedding:",
        item["embedding"].shape
    )

MULTI-IMAGE WARDROBE TEST

Number of wardrobe items: 3

Item 1
------------------------------
Name: WOMEN-Cardigans-id_00005032-03_4_full.png
Category: Cardigans
Confidence: 55.70%
Embedding: (768,)

Item 2
------------------------------
Name: WOMEN-Blouses_Shirts-id_00002244-01_4_full.png
Category: Blouses_Shirts
Confidence: 69.90%
Embedding: (768,)

Item 3
------------------------------
Name: WOMEN-Dresses-id_00003920-02_7_additional.png
Category: Dresses
Confidence: 83.24%
Embedding: (768,)


In [ ]:
# --------------------------------------------------
# Generate outfits from real wardrobe
# --------------------------------------------------

real_generated_outfits = (
    generate_outfit_combinations(
        test_wardrobe_items
    )
)


print("=" * 55)
print("REAL WARDROBE OUTFIT GENERATION")
print("=" * 55)

print(
    "Generated outfits:",
    len(real_generated_outfits)
)


# --------------------------------------------------
# Score every outfit
# --------------------------------------------------

real_scored_outfits = []

for outfit in real_generated_outfits:

    result = score_outfit(
        outfit=outfit,

        mood="Confident",

        personality="Extrovert",

        occasion="Interview",

        confidence=8,

        user_preference=user_preference_vector
    )

    real_scored_outfits.append(result)


# --------------------------------------------------
# Rank outfits
# --------------------------------------------------

real_ranked_outfits = rank_outfits(
    real_scored_outfits
)


# --------------------------------------------------
# Display ranking
# --------------------------------------------------

print("\n" + "=" * 55)
print("REAL WARDROBE RECOMMENDATIONS")
print("=" * 55)


for rank, result in enumerate(
    real_ranked_outfits,
    start=1
):

    names = [
        item["name"]
        for item in result["items"]
    ]

    print(f"\nRank {rank}")
    print("-" * 30)

    print(
        "Outfit:",
        " + ".join(names)
    )

    print(
        "Categories:",
        " + ".join(result["categories"])
    )

    print(
        "Compatibility:",
        f"{result['compatibility_score']:.2f}%"
    )

    print(
        "Psychology:",
        f"{result['psychology_score']:.2f}%"
    )

    print(
        "Preference:",
        f"{result['preference_score']:.2f}%"
    )

    print(
        "Final Score:",
        f"{result['final_score']:.2f}%"
    )

REAL WARDROBE OUTFIT GENERATION
Generated outfits: 2

REAL WARDROBE RECOMMENDATIONS

Rank 1
------------------------------
Outfit: WOMEN-Dresses-id_00003920-02_7_additional.png + WOMEN-Cardigans-id_00005032-03_4_full.png
Categories: Dresses + Cardigans
Compatibility: 39.07%
Psychology: 81.65%
Preference: 50.00%
Final Score: 54.03%

Rank 2
------------------------------
Outfit: WOMEN-Dresses-id_00003920-02_7_additional.png
Categories: Dresses
Compatibility: 25.63%
Psychology: 90.20%
Preference: 50.00%
Final Score: 49.87%


In [ ]:
def generate_outfit_explanation(
    result,
    mood,
    personality,
    occasion,
    confidence
):
    """
    Generate a transparent explanation
    for a PsyStyle recommendation.
    """

    explanations = []

    compatibility = result["compatibility_score"]
    psychology = result["psychology_score"]
    preference = result["preference_score"]


    # --------------------------------------------------
    # Compatibility explanation
    # --------------------------------------------------

    if compatibility >= 75:

        explanations.append(
            "The clothing items have strong "
            "visual compatibility."
        )

    elif compatibility >= 50:

        explanations.append(
            "The clothing items have moderate "
            "visual compatibility."
        )

    else:

        explanations.append(
            "The clothing items have lower "
            "visual compatibility."
        )


    # --------------------------------------------------
    # Psychology explanation
    # --------------------------------------------------

    if psychology >= 80:

        explanations.append(
            f"The outfit strongly matches your "
            f"{mood.lower()} mood, "
            f"{personality.lower()} personality, "
            f"and {occasion.lower()} context."
        )

    elif psychology >= 60:

        explanations.append(
            f"The outfit reasonably matches your "
            f"{mood.lower()} mood and "
            f"{occasion.lower()} context."
        )

    else:

        explanations.append(
            f"The outfit has a weaker match with "
            f"your selected psychological context."
        )


    # --------------------------------------------------
    # Confidence explanation
    # --------------------------------------------------

    if confidence >= 7:

        explanations.append(
            f"Your confidence level of {confidence}/10 "
            f"supports this style choice."
        )

    elif confidence <= 4:

        explanations.append(
            f"Your confidence level is {confidence}/10, "
            f"so the recommendation considers a "
            f"more comfortable style."
        )


    # --------------------------------------------------
    # Preference explanation
    # --------------------------------------------------

    if preference > 60:

        explanations.append(
            "The outfit is aligned with your "
            "previous preferences."
        )

    elif preference < 40:

        explanations.append(
            "The outfit differs from your "
            "previous preferences."
        )

    else:

        explanations.append(
            "There is currently limited preference "
            "feedback from the user."
        )


    return explanations


print("=" * 55)
print("OUTFIT EXPLANATION SYSTEM CREATED")
print("=" * 55)

print("Explanation factors:")
print("  • Compatibility")
print("  • Mood")
print("  • Personality")
print("  • Occasion")
print("  • Confidence")
print("  • User preferences")

OUTFIT EXPLANATION SYSTEM CREATED
Explanation factors:
  • Compatibility
  • Mood
  • Personality
  • Occasion
  • Confidence
  • User preferences


In [ ]:
# --------------------------------------------------
# Select the top-ranked outfit
# --------------------------------------------------

top_outfit = real_ranked_outfits[0]


# --------------------------------------------------
# Generate explanation
# --------------------------------------------------

top_explanation = generate_outfit_explanation(
    result=top_outfit,
    mood="Confident",
    personality="Extrovert",
    occasion="Interview",
    confidence=8
)


# --------------------------------------------------
# Display recommendation
# --------------------------------------------------

print("=" * 55)
print("PSYSTYLE RECOMMENDATION")
print("=" * 55)


names = [
    item["name"]
    for item in top_outfit["items"]
]


print("\nRecommended Outfit:")
print(
    " + ".join(names)
)


print("\nCategories:")
print(
    " + ".join(top_outfit["categories"])
)


print("\nScores")
print("-" * 30)

print(
    "Compatibility:",
    f"{top_outfit['compatibility_score']:.2f}%"
)

print(
    "Psychology:",
    f"{top_outfit['psychology_score']:.2f}%"
)

print(
    "Preference:",
    f"{top_outfit['preference_score']:.2f}%"
)

print(
    "Final Score:",
    f"{top_outfit['final_score']:.2f}%"
)


print("\nWhy this outfit?")
print("-" * 30)

for explanation in top_explanation:
    print("✓", explanation)

PSYSTYLE RECOMMENDATION

Recommended Outfit:
WOMEN-Dresses-id_00003920-02_7_additional.png + WOMEN-Cardigans-id_00005032-03_4_full.png

Categories:
Dresses + Cardigans

Scores
------------------------------
Compatibility: 39.07%
Psychology: 81.65%
Preference: 50.00%
Final Score: 54.03%

Why this outfit?
------------------------------
✓ The clothing items have lower visual compatibility.
✓ The outfit strongly matches your confident mood, extrovert personality, and interview context.
✓ Your confidence level of 8/10 supports this style choice.
✓ There is currently limited preference feedback from the user.


In [ ]:
def process_user_feedback(
    result,
    feedback,
    current_preference
):
    """
    Process user feedback for a recommended outfit.

    feedback:
        "like"
        "dislike"
        "skip"

    Returns:
        updated preference vector
    """

    # --------------------------------------------------
    # Get outfit embeddings
    # --------------------------------------------------

    outfit_embeddings = [
        item["embedding"]
        for item in result["items"]
    ]


    # --------------------------------------------------
    # Convert outfit into preference representation
    # --------------------------------------------------

    outfit_preference = (
        encode_outfit_preference(
            outfit_embeddings
        )
    )


    # --------------------------------------------------
    # Update user preference
    # --------------------------------------------------

    updated_preference = (
        update_preference_vector(
            current_preference=current_preference,
            outfit_preference=outfit_preference,
            feedback=feedback
        )
    )


    return updated_preference


print("=" * 55)
print("USER FEEDBACK SYSTEM CREATED")
print("=" * 55)

print("❤️ Like    → Positive preference update")
print("👎 Dislike → Negative preference update")
print("— Skip     → No preference update")


USER FEEDBACK SYSTEM CREATED
❤️ Like    → Positive preference update
👎 Dislike → Negative preference update
— Skip     → No preference update


In [ ]:
# --------------------------------------------------
# Current preference before feedback
# --------------------------------------------------

preference_before = user_preference_vector.clone()


print("=" * 55)
print("LIKE FEEDBACK TEST")
print("=" * 55)

print(
    "\nPreference norm BEFORE:",
    f"{preference_before.norm().item():.4f}"
)


# --------------------------------------------------
# Simulate ❤️ Like
# --------------------------------------------------

user_preference_vector = process_user_feedback(
    result=top_outfit,
    feedback="like",
    current_preference=user_preference_vector
)


# --------------------------------------------------
# Check updated preference
# --------------------------------------------------

print(
    "Preference norm AFTER:",
    f"{user_preference_vector.norm().item():.4f}"
)


print(
    "\nPreference vector updated:",
    not torch.equal(
        preference_before,
        user_preference_vector
    )
)


print("\nFeedback:")
print("❤️ Like → Preference vector updated")

LIKE FEEDBACK TEST

Preference norm BEFORE: 0.0000
Preference norm AFTER: 1.0000

Preference vector updated: True

Feedback:
❤️ Like → Preference vector updated


In [ ]:
# --------------------------------------------------
# Encode the same outfit
# --------------------------------------------------

liked_outfit_preference = (
    encode_outfit_preference(
        [
            item["embedding"]
            for item in top_outfit["items"]
        ]
    )
)


# --------------------------------------------------
# Calculate personalized preference score
# --------------------------------------------------

personalized_preference_score = (
    calculate_preference_score(
        user_preference=user_preference_vector,
        outfit_preference=liked_outfit_preference
    )
)


# --------------------------------------------------
# Display result
# --------------------------------------------------

print("=" * 55)
print("PERSONALIZED PREFERENCE TEST")
print("=" * 55)

print(
    "\nPrevious preference score:",
    "50.00%"
)

print(
    "New preference score:",
    f"{personalized_preference_score:.2f}%"
)

print(
    "\nPreference is now personalized:",
    personalized_preference_score != 50.0
)

PERSONALIZED PREFERENCE TEST

Previous preference score: 50.00%
New preference score: 100.00%

Preference is now personalized: True


In [ ]:
# --------------------------------------------------
# Re-score all generated outfits
# using the updated preference vector
# --------------------------------------------------

personalized_scored_outfits = []

for outfit in real_generated_outfits:

    result = score_outfit(
        outfit=outfit,

        mood="Confident",

        personality="Extrovert",

        occasion="Interview",

        confidence=8,

        user_preference=user_preference_vector
    )

    personalized_scored_outfits.append(result)


# --------------------------------------------------
# Rank using personalized scores
# --------------------------------------------------

personalized_ranked_outfits = rank_outfits(
    personalized_scored_outfits
)


# --------------------------------------------------
# Display personalized ranking
# --------------------------------------------------

print("=" * 55)
print("PERSONALIZED PSYSTYLE RANKING")
print("=" * 55)


for rank, result in enumerate(
    personalized_ranked_outfits,
    start=1
):

    names = [
        item["name"]
        for item in result["items"]
    ]

    print(f"\nRank {rank}")
    print("-" * 30)

    print(
        "Outfit:",
        " + ".join(names)
    )

    print(
        "Compatibility:",
        f"{result['compatibility_score']:.2f}%"
    )

    print(
        "Psychology:",
        f"{result['psychology_score']:.2f}%"
    )

    print(
        "Preference:",
        f"{result['preference_score']:.2f}%"
    )

    print(
        "Final Score:",
        f"{result['final_score']:.2f}%"
    )

PERSONALIZED PSYSTYLE RANKING

Rank 1
------------------------------
Outfit: WOMEN-Dresses-id_00003920-02_7_additional.png + WOMEN-Cardigans-id_00005032-03_4_full.png
Compatibility: 39.07%
Psychology: 81.65%
Preference: 100.00%
Final Score: 64.03%

Rank 2
------------------------------
Outfit: WOMEN-Dresses-id_00003920-02_7_additional.png
Compatibility: 25.63%
Psychology: 90.20%
Preference: 90.12%
Final Score: 57.90%


In [ ]:
import os

STREAMLIT_DIR = "/content/drive/MyDrive/PsyStyle/streamlit"

os.makedirs(
    STREAMLIT_DIR,
    exist_ok=True
)

APP_PATH = os.path.join(
    STREAMLIT_DIR,
    "app.py"
)

print("=" * 55)
print("STREAMLIT APP LOCATION CREATED")
print("=" * 55)

print("\nApp path:")
print(APP_PATH)

print("\nThis file will contain the final PsyStyle application.")
print("No model training will be performed by the app.")

STREAMLIT APP LOCATION CREATED

App path:
/content/drive/MyDrive/PsyStyle/streamlit/app.py

This file will contain the final PsyStyle application.
No model training will be performed by the app.


In [ ]:
APP_CODE = r'''
import os
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from PIL import Image
from transformers import ViTModel, ViTImageProcessor


# ==================================================
# PATHS
# ==================================================

BASE_DIR = "/content/drive/MyDrive/PsyStyle"

VIT_MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "vit",
    "psystyle_vit_fashion.pt"
)

COMPATIBILITY_MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "compatibility",
    "psystyle_compatibility.pt"
)

EMBEDDING_PATH = os.path.join(
    BASE_DIR,
    "models",
    "compatibility",
    "polyvore_vit_embeddings.npz"
)


# ==================================================
# DEVICE
# ==================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ==================================================
# STREAMLIT PAGE CONFIGURATION
# ==================================================

st.set_page_config(
    page_title="PsyStyle",
    page_icon="👗",
    layout="wide"
)


# ==================================================
# APPLICATION TITLE
# ==================================================

st.title("👗 PsyStyle")

st.subheader(
    "Deep Learning-Based Psychological "
    "Outfit Recommendation System"
)

st.write(
    "Upload your wardrobe and let PsyStyle "
    "recommend outfits based on visual compatibility, "
    "psychological context, and user preferences."
)
'''

with open(APP_PATH, "w") as f:
    f.write(APP_CODE)


print("=" * 55)
print("STREAMLIT APP FOUNDATION CREATED")
print("=" * 55)

print("\nSaved to:")
print(APP_PATH)

print("\nIncluded:")
print("  ✓ Required imports")
print("  ✓ Drive model paths")
print("  ✓ Device configuration")
print("  ✓ Streamlit page configuration")
print("  ✓ PsyStyle title and description")

STREAMLIT APP FOUNDATION CREATED

Saved to:
/content/drive/MyDrive/PsyStyle/streamlit/app.py

Included:
  ✓ Required imports
  ✓ Drive model paths
  ✓ Device configuration
  ✓ Streamlit page configuration
  ✓ PsyStyle title and description


In [ ]:
APP_CODE = r'''
# ==================================================
# VIT MODEL
# ==================================================

CLASS_NAMES = [
    "Tees_Tanks",
    "Dresses",
    "Blouses_Shirts",
    "Sweaters",
    "Shorts",
    "Pants",
    "Rompers_Jumpsuits",
    "Jackets_Coats",
    "Cardigans",
    "Skirts",
    "Sweatshirts_Hoodies",
    "Graphic_Tees",
    "Denim",
    "Shirts_Polos",
    "Jackets_Vests",
    "Leggings"
]


class PsyStyleViT(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.classifier = nn.Linear(
            self.vit.config.hidden_size,
            num_classes
        )


    def forward(
        self,
        pixel_values,
        return_embedding=False
    ):

        outputs = self.vit(
            pixel_values=pixel_values
        )

        embedding = (
            outputs.last_hidden_state[:, 0, :]
        )

        logits = self.classifier(
            embedding
        )

        if return_embedding:

            return logits, embedding

        return logits


@st.cache_resource
def load_vit_model():

    checkpoint = torch.load(
        VIT_MODEL_PATH,
        map_location=DEVICE
    )

    model = PsyStyleViT(
        num_classes=len(CLASS_NAMES)
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(DEVICE)

    model.eval()

    return model


@st.cache_resource
def load_vit_processor():

    return ViTImageProcessor.from_pretrained(
        "google/vit-base-patch16-224"
    )


vit_model = load_vit_model()

processor = load_vit_processor()
'''

# Read the existing app
with open(APP_PATH, "r") as f:
    existing_code = f.read()

# Append the ViT section
with open(APP_PATH, "w") as f:
    f.write(existing_code)
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("VIT MODEL LOADER ADDED")
print("=" * 55)

print("\nAdded:")
print("  ✓ PsyStyleViT architecture")
print("  ✓ Saved model loading")
print("  ✓ ViT processor loading")
print("  ✓ Streamlit model caching")
print("  ✓ GPU/CPU support")

print("\nTraining:")
print("  ✗ No training")
print("  ✓ Saved model is loaded")

VIT MODEL LOADER ADDED

Added:
  ✓ PsyStyleViT architecture
  ✓ Saved model loading
  ✓ ViT processor loading
  ✓ Streamlit model caching
  ✓ GPU/CPU support

Training:
  ✗ No training
  ✓ Saved model is loaded


In [ ]:
APP_CODE = r'''
# ==================================================
# VIT IMAGE PREDICTION
# ==================================================

def predict_clothing_item(image):

    image = image.convert("RGB")

    processed = processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = (
        processed["pixel_values"]
        .to(DEVICE)
    )

    with torch.no_grad():

        logits, embedding = vit_model(
            pixel_values,
            return_embedding=True
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predicted_index = torch.argmax(
            probabilities,
            dim=1
        ).item()

        confidence = probabilities[
            0,
            predicted_index
        ].item()

    category = CLASS_NAMES[
        predicted_index
    ]

    embedding = (
        embedding
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return (
        category,
        confidence,
        embedding
    )
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("VIT IMAGE PREDICTION ADDED")
print("=" * 55)

print("\nThe Streamlit app can now:")
print("  ✓ Accept a clothing image")
print("  ✓ Classify the clothing")
print("  ✓ Calculate confidence")
print("  ✓ Generate a 768-D embedding")

VIT IMAGE PREDICTION ADDED

The Streamlit app can now:
  ✓ Accept a clothing image
  ✓ Classify the clothing
  ✓ Calculate confidence
  ✓ Generate a 768-D embedding


In [ ]:
APP_CODE = r'''
# ==================================================
# COMPATIBILITY NETWORK
# ==================================================

class CompatibilityNetwork(nn.Module):

    def __init__(
        self,
        embedding_dim=768
    ):

        super().__init__()

        self.item_encoder = nn.Sequential(
            nn.Linear(
                embedding_dim,
                256
            ),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.compatibility_network = nn.Sequential(
            nn.Linear(
                256,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                128,
                1
            )
        )


    def masked_mean_pool(
        self,
        item_embeddings,
        mask
    ):

        mask = mask.unsqueeze(-1).float()

        masked_embeddings = (
            item_embeddings * mask
        )

        summed = masked_embeddings.sum(
            dim=1
        )

        count = mask.sum(
            dim=1
        ).clamp(min=1)

        return summed / count


    def forward(
        self,
        embeddings,
        mask
    ):

        encoded_items = self.item_encoder(
            embeddings
        )

        outfit_embedding = (
            self.masked_mean_pool(
                encoded_items,
                mask
            )
        )

        logits = self.compatibility_network(
            outfit_embedding
        )

        return logits.squeeze(-1)


@st.cache_resource
def load_compatibility_model():

    checkpoint = torch.load(
        COMPATIBILITY_MODEL_PATH,
        map_location=DEVICE
    )

    model = CompatibilityNetwork(
        embedding_dim=checkpoint[
            "embedding_size"
        ]
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model = model.to(DEVICE)

    model.eval()

    return model


compatibility_model = (
    load_compatibility_model()
)


def calculate_compatibility_score(
    item_embeddings
):

    embeddings = np.asarray(
        item_embeddings,
        dtype=np.float32
    )

    embeddings = torch.from_numpy(
        embeddings
    )

    embeddings = embeddings.unsqueeze(0)

    num_items = embeddings.shape[1]

    mask = torch.ones(
        1,
        num_items,
        dtype=torch.bool
    )

    embeddings = embeddings.to(
        DEVICE
    )

    mask = mask.to(
        DEVICE
    )

    with torch.no_grad():

        logits = compatibility_model(
            embeddings,
            mask
        )

        probability = torch.sigmoid(
            logits
        ).item()

    return probability * 100
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("COMPATIBILITY NETWORK ADDED")
print("=" * 55)

print("\nAdded:")
print("  ✓ Saved compatibility model")
print("  ✓ Model caching")
print("  ✓ 768-D embedding input")
print("  ✓ Compatibility score calculation")
print("  ✓ GPU/CPU support")

print("\nTraining:")
print("  ✗ No training")
print("  ✓ Best saved model is loaded")

COMPATIBILITY NETWORK ADDED

Added:
  ✓ Saved compatibility model
  ✓ Model caching
  ✓ 768-D embedding input
  ✓ Compatibility score calculation
  ✓ GPU/CPU support

Training:
  ✗ No training
  ✓ Best saved model is loaded


In [ ]:
APP_CODE = r'''
# ==================================================
# PSYCHOLOGY RULES
# ==================================================

PSYCHOLOGY_RULES = {

    "Happy": {
        "Tees_Tanks": 0.90,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.75,
        "Sweaters": 0.70,
        "Shorts": 0.90,
        "Pants": 0.70,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.65,
        "Cardigans": 0.70,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.80,
        "Graphic_Tees": 0.95,
        "Denim": 0.85,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.70,
        "Leggings": 0.75
    },

    "Nervous": {
        "Tees_Tanks": 0.65,
        "Dresses": 0.60,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.85,
        "Shorts": 0.50,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.55,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.90,
        "Skirts": 0.60,
        "Sweatshirts_Hoodies": 0.85,
        "Graphic_Tees": 0.55,
        "Denim": 0.80,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.80,
        "Leggings": 0.75
    },

    "Confident": {
        "Tees_Tanks": 0.75,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.90,
        "Sweaters": 0.75,
        "Shorts": 0.75,
        "Pants": 0.90,
        "Rompers_Jumpsuits": 0.85,
        "Jackets_Coats": 0.95,
        "Cardigans": 0.70,
        "Skirts": 0.85,
        "Sweatshirts_Hoodies": 0.70,
        "Graphic_Tees": 0.80,
        "Denim": 0.85,
        "Shirts_Polos": 0.90,
        "Jackets_Vests": 0.90,
        "Leggings": 0.70
    }
}


PERSONALITY_RULES = {

    "Introvert": {
        "Tees_Tanks": 0.75,
        "Dresses": 0.70,
        "Blouses_Shirts": 0.85,
        "Sweaters": 0.90,
        "Shorts": 0.65,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.65,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.95,
        "Skirts": 0.70,
        "Sweatshirts_Hoodies": 0.90,
        "Graphic_Tees": 0.55,
        "Denim": 0.80,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.75,
        "Leggings": 0.80
    },

    "Extrovert": {
        "Tees_Tanks": 0.85,
        "Dresses": 0.90,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.70,
        "Shorts": 0.85,
        "Pants": 0.75,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.85,
        "Cardigans": 0.65,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.70,
        "Graphic_Tees": 0.95,
        "Denim": 0.90,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.90,
        "Leggings": 0.70
    }
}


OCCASION_RULES = {

    "Interview": {
        "Tees_Tanks": 0.35,
        "Dresses": 0.80,
        "Blouses_Shirts": 0.95,
        "Sweaters": 0.75,
        "Shorts": 0.10,
        "Pants": 0.95,
        "Rompers_Jumpsuits": 0.50,
        "Jackets_Coats": 0.90,
        "Cardigans": 0.80,
        "Skirts": 0.85,
        "Sweatshirts_Hoodies": 0.20,
        "Graphic_Tees": 0.10,
        "Denim": 0.50,
        "Shirts_Polos": 0.90,
        "Jackets_Vests": 0.85,
        "Leggings": 0.25
    },

    "Party": {
        "Tees_Tanks": 0.80,
        "Dresses": 0.95,
        "Blouses_Shirts": 0.80,
        "Sweaters": 0.65,
        "Shorts": 0.85,
        "Pants": 0.75,
        "Rompers_Jumpsuits": 0.90,
        "Jackets_Coats": 0.80,
        "Cardigans": 0.55,
        "Skirts": 0.95,
        "Sweatshirts_Hoodies": 0.55,
        "Graphic_Tees": 0.90,
        "Denim": 0.85,
        "Shirts_Polos": 0.80,
        "Jackets_Vests": 0.90,
        "Leggings": 0.65
    },

    "College": {
        "Tees_Tanks": 0.90,
        "Dresses": 0.70,
        "Blouses_Shirts": 0.75,
        "Sweaters": 0.80,
        "Shorts": 0.85,
        "Pants": 0.85,
        "Rompers_Jumpsuits": 0.75,
        "Jackets_Coats": 0.75,
        "Cardigans": 0.70,
        "Skirts": 0.70,
        "Sweatshirts_Hoodies": 0.90,
        "Graphic_Tees": 0.90,
        "Denim": 0.95,
        "Shirts_Polos": 0.85,
        "Jackets_Vests": 0.80,
        "Leggings": 0.75
    },

    "Wedding": {
        "Tees_Tanks": 0.20,
        "Dresses": 0.95,
        "Blouses_Shirts": 0.85,
        "Sweaters": 0.45,
        "Shorts": 0.05,
        "Pants": 0.70,
        "Rompers_Jumpsuits": 0.80,
        "Jackets_Coats": 0.70,
        "Cardigans": 0.45,
        "Skirts": 0.90,
        "Sweatshirts_Hoodies": 0.05,
        "Graphic_Tees": 0.05,
        "Denim": 0.15,
        "Shirts_Polos": 0.60,
        "Jackets_Vests": 0.65,
        "Leggings": 0.10
    }
}


# ==================================================
# PSYCHOLOGY SCORE
# ==================================================

def calculate_psychology_score(
    category,
    mood,
    personality,
    occasion,
    confidence
):

    mood_score = (
        PSYCHOLOGY_RULES[mood][category]
    )

    personality_score = (
        PERSONALITY_RULES[personality][category]
    )

    occasion_score = (
        OCCASION_RULES[occasion][category]
    )

    confidence_normalized = (
        confidence / 10.0
    )

    base_score = (
        0.40 * mood_score
        +
        0.30 * personality_score
        +
        0.30 * occasion_score
    )

    confidence_adjustment = (
        0.10
        *
        confidence_normalized
        *
        (mood_score - 0.5)
    )

    final_score = (
        base_score
        +
        confidence_adjustment
    )

    final_score = max(
        0.0,
        min(1.0, final_score)
    )

    return final_score


def calculate_outfit_psychology_score(
    categories,
    mood,
    personality,
    occasion,
    confidence
):

    item_scores = []

    for category in categories:

        score = calculate_psychology_score(
            category,
            mood,
            personality,
            occasion,
            confidence
        )

        item_scores.append(score)

    outfit_score = (
        sum(item_scores)
        /
        len(item_scores)
    )

    return outfit_score * 100
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("PSYCHOLOGY MODULE ADDED")
print("=" * 55)

print("\nAdded:")
print("  ✓ Mood rules")
print("  ✓ Personality rules")
print("  ✓ Occasion rules")
print("  ✓ Confidence adjustment")
print("  ✓ Item psychology score")
print("  ✓ Outfit psychology score")

print("\nPsychology module:")
print("  Rule-based")
print("  Not separately trained")

PSYCHOLOGY MODULE ADDED

Added:
  ✓ Mood rules
  ✓ Personality rules
  ✓ Occasion rules
  ✓ Confidence adjustment
  ✓ Item psychology score
  ✓ Outfit psychology score

Psychology module:
  Rule-based
  Not separately trained


In [ ]:
APP_CODE = r'''
# ==================================================
# PREFERENCE ENCODER
# ==================================================

class PreferenceEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        preference_dim=256
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                embedding_dim,
                512
            ),

            nn.ReLU(),

            nn.Linear(
                512,
                preference_dim
            ),

            nn.ReLU()
        )


    def forward(
        self,
        outfit_embedding
    ):

        return self.encoder(
            outfit_embedding
        )


@st.cache_resource
def load_preference_encoder():

    model = PreferenceEncoder(
        embedding_dim=768,
        preference_dim=256
    )

    model = model.to(DEVICE)

    model.eval()

    return model


preference_encoder = (
    load_preference_encoder()
)


PREFERENCE_DIM = 256


# ==================================================
# USER PREFERENCE VECTOR
# ==================================================

if "user_preference_vector" not in st.session_state:

    st.session_state.user_preference_vector = (
        torch.zeros(
            PREFERENCE_DIM,
            dtype=torch.float32,
            device=DEVICE
        )
    )


# ==================================================
# OUTFIT REPRESENTATION
# ==================================================

def create_outfit_representation(
    item_embeddings
):

    embeddings = torch.tensor(
        np.asarray(
            item_embeddings,
            dtype=np.float32
        ),
        dtype=torch.float32,
        device=DEVICE
    )

    outfit_embedding = (
        embeddings.mean(dim=0)
    )

    return outfit_embedding


def encode_outfit_preference(
    item_embeddings
):

    outfit_embedding = (
        create_outfit_representation(
            item_embeddings
        )
    )

    outfit_embedding = (
        outfit_embedding.unsqueeze(0)
    )

    with torch.no_grad():

        preference_vector = (
            preference_encoder(
                outfit_embedding
            )
        )

    preference_vector = F.normalize(
        preference_vector,
        p=2,
        dim=1
    )

    return preference_vector.squeeze(0)


# ==================================================
# PREFERENCE SCORE
# ==================================================

def calculate_preference_score(
    user_preference,
    outfit_preference
):

    if torch.norm(
        user_preference
    ) < 1e-8:

        return 50.0


    user_vector = F.normalize(
        user_preference.unsqueeze(0),
        p=2,
        dim=1
    )

    outfit_vector = F.normalize(
        outfit_preference.unsqueeze(0),
        p=2,
        dim=1
    )

    similarity = F.cosine_similarity(
        user_vector,
        outfit_vector,
        dim=1
    ).item()

    score = (
        (similarity + 1.0)
        / 2.0
        * 100.0
    )

    return score


# ==================================================
# UPDATE USER PREFERENCE
# ==================================================

def update_preference_vector(
    current_preference,
    outfit_preference,
    feedback,
    learning_rate=0.20
):

    if feedback == "like":

        updated_preference = (
            current_preference
            +
            learning_rate
            * outfit_preference
        )

    elif feedback == "dislike":

        updated_preference = (
            current_preference
            -
            learning_rate
            * outfit_preference
        )

    elif feedback == "skip":

        updated_preference = (
            current_preference
        )

    else:

        raise ValueError(
            "Invalid feedback."
        )


    if torch.norm(
        updated_preference
    ) > 1e-8:

        updated_preference = F.normalize(
            updated_preference,
            p=2,
            dim=0
        )


    return updated_preference


# ==================================================
# PROCESS USER FEEDBACK
# ==================================================

def process_user_feedback(
    result,
    feedback
):

    outfit_embeddings = [
        item["embedding"]
        for item in result["items"]
    ]

    outfit_preference = (
        encode_outfit_preference(
            outfit_embeddings
        )
    )

    current_preference = (
        st.session_state.user_preference_vector
    )

    updated_preference = (
        update_preference_vector(
            current_preference,
            outfit_preference,
            feedback
        )
    )

    st.session_state.user_preference_vector = (
        updated_preference
    )
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("PREFERENCE SYSTEM ADDED")
print("=" * 55)

print("\nAdded:")
print("  ✓ Preference Encoder")
print("  ✓ 256-D preference vector")
print("  ✓ Session-based user preference")
print("  ✓ Like update")
print("  ✓ Dislike update")
print("  ✓ Skip = no update")
print("  ✓ Preference scoring")

PREFERENCE SYSTEM ADDED

Added:
  ✓ Preference Encoder
  ✓ 256-D preference vector
  ✓ Session-based user preference
  ✓ Like update
  ✓ Dislike update
  ✓ Skip = no update
  ✓ Preference scoring


In [ ]:
APP_CODE = r'''
# ==================================================
# OUTFIT RULES
# ==================================================

OUTFIT_RULES = {

    "top": [
        "Tees_Tanks",
        "Blouses_Shirts",
        "Sweaters",
        "Sweatshirts_Hoodies",
        "Graphic_Tees",
        "Shirts_Polos"
    ],

    "bottom": [
        "Pants",
        "Shorts",
        "Denim",
        "Skirts",
        "Leggings"
    ],

    "outerwear": [
        "Jackets_Coats",
        "Cardigans",
        "Jackets_Vests"
    ],

    "one_piece": [
        "Dresses",
        "Rompers_Jumpsuits"
    ]
}


def get_outfit_role(category):

    for role, categories in (
        OUTFIT_RULES.items()
    ):

        if category in categories:
            return role

    return None


# ==================================================
# GENERATE OUTFIT COMBINATIONS
# ==================================================

def generate_outfit_combinations(
    wardrobe_items
):

    outfits = []

    tops = []
    bottoms = []
    outerwear = []
    one_pieces = []


    for item in wardrobe_items:

        role = get_outfit_role(
            item["category"]
        )

        if role == "top":
            tops.append(item)

        elif role == "bottom":
            bottoms.append(item)

        elif role == "outerwear":
            outerwear.append(item)

        elif role == "one_piece":
            one_pieces.append(item)


    # Top + Bottom

    for top in tops:

        for bottom in bottoms:

            outfits.append([
                top,
                bottom
            ])


    # Top + Bottom + Outerwear

    for top in tops:

        for bottom in bottoms:

            for outer in outerwear:

                outfits.append([
                    top,
                    bottom,
                    outer
                ])


    # One-piece

    for one_piece in one_pieces:

        outfits.append([
            one_piece
        ])


        # One-piece + Outerwear

        for outer in outerwear:

            outfits.append([
                one_piece,
                outer
            ])


    return outfits


# ==================================================
# FINAL SCORE
# ==================================================

def calculate_final_outfit_score(
    compatibility_score,
    psychology_score,
    preference_score
):

    compatibility_weight = 0.50
    psychology_weight = 0.30
    preference_weight = 0.20


    final_score = (

        compatibility_weight
        * compatibility_score

        +

        psychology_weight
        * psychology_score

        +

        preference_weight
        * preference_score
    )


    return final_score


# ==================================================
# SCORE ONE OUTFIT
# ==================================================

def score_outfit(
    outfit,
    mood,
    personality,
    occasion,
    confidence,
    user_preference
):

    embeddings = [
        item["embedding"]
        for item in outfit
    ]

    categories = [
        item["category"]
        for item in outfit
    ]


    # ------------------------------
    # Compatibility
    # ------------------------------

    compatibility_score = (
        calculate_compatibility_score(
            embeddings
        )
    )


    # ------------------------------
    # Psychology
    # ------------------------------

    psychology_score = (
        calculate_outfit_psychology_score(
            categories,
            mood,
            personality,
            occasion,
            confidence
        )
    )


    # ------------------------------
    # Preference
    # ------------------------------

    outfit_preference = (
        encode_outfit_preference(
            embeddings
        )
    )

    preference_score = (
        calculate_preference_score(
            user_preference,
            outfit_preference
        )
    )


    # ------------------------------
    # Final Score
    # ------------------------------

    final_score = (
        calculate_final_outfit_score(
            compatibility_score,
            psychology_score,
            preference_score
        )
    )


    return {

        "items": outfit,

        "categories": categories,

        "compatibility_score":
            compatibility_score,

        "psychology_score":
            psychology_score,

        "preference_score":
            preference_score,

        "final_score":
            final_score
    }


# ==================================================
# RANK OUTFITS
# ==================================================

def rank_outfits(
    scored_outfits
):

    return sorted(
        scored_outfits,
        key=lambda outfit:
            outfit["final_score"],
        reverse=True
    )
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("RECOMMENDATION ENGINE ADDED")
print("=" * 55)

print("\nAdded:")
print("  ✓ Wardrobe outfit generation")
print("  ✓ Top + Bottom")
print("  ✓ Top + Bottom + Outerwear")
print("  ✓ One-piece outfits")
print("  ✓ One-piece + Outerwear")
print("  ✓ Compatibility scoring")
print("  ✓ Psychology scoring")
print("  ✓ Preference scoring")
print("  ✓ Final weighted score")
print("  ✓ Outfit ranking")

print("\nFinal score weights:")
print("  Compatibility : 50%")
print("  Psychology    : 30%")
print("  Preference    : 20%")

RECOMMENDATION ENGINE ADDED

Added:
  ✓ Wardrobe outfit generation
  ✓ Top + Bottom
  ✓ Top + Bottom + Outerwear
  ✓ One-piece outfits
  ✓ One-piece + Outerwear
  ✓ Compatibility scoring
  ✓ Psychology scoring
  ✓ Preference scoring
  ✓ Final weighted score
  ✓ Outfit ranking

Final score weights:
  Compatibility : 50%
  Psychology    : 30%
  Preference    : 20%


In [ ]:
APP_CODE = r'''
# ==================================================
# OUTFIT EXPLANATION
# ==================================================

def generate_outfit_explanation(
    result,
    mood,
    personality,
    occasion,
    confidence
):

    explanations = []

    compatibility = (
        result["compatibility_score"]
    )

    psychology = (
        result["psychology_score"]
    )

    preference = (
        result["preference_score"]
    )


    # ------------------------------
    # Compatibility explanation
    # ------------------------------

    if compatibility >= 75:

        explanations.append(
            "The clothing items have strong "
            "visual compatibility."
        )

    elif compatibility >= 50:

        explanations.append(
            "The clothing items have moderate "
            "visual compatibility."
        )

    else:

        explanations.append(
            "The clothing items have lower "
            "visual compatibility."
        )


    # ------------------------------
    # Psychology explanation
    # ------------------------------

    if psychology >= 80:

        explanations.append(
            f"The outfit strongly matches your "
            f"{mood.lower()} mood, "
            f"{personality.lower()} personality, "
            f"and {occasion.lower()} context."
        )

    elif psychology >= 60:

        explanations.append(
            f"The outfit reasonably matches your "
            f"{mood.lower()} mood and "
            f"{occasion.lower()} context."
        )

    else:

        explanations.append(
            "The outfit has a weaker match with "
            "your selected psychological context."
        )


    # ------------------------------
    # Confidence explanation
    # ------------------------------

    if confidence >= 7:

        explanations.append(
            f"Your confidence level of "
            f"{confidence}/10 supports this "
            f"style choice."
        )

    elif confidence <= 4:

        explanations.append(
            f"Your confidence level is "
            f"{confidence}/10, so the recommendation "
            f"considers a more comfortable style."
        )


    # ------------------------------
    # Preference explanation
    # ------------------------------

    if preference > 60:

        explanations.append(
            "The outfit is aligned with your "
            "previous preferences."
        )

    elif preference < 40:

        explanations.append(
            "The outfit differs from your "
            "previous preferences."
        )

    else:

        explanations.append(
            "There is currently limited preference "
            "feedback from the user."
        )


    return explanations
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("EXPLANATION SYSTEM ADDED")
print("=" * 55)

print("\nThe app can now explain:")
print("  ✓ Visual compatibility")
print("  ✓ Mood")
print("  ✓ Personality")
print("  ✓ Occasion")
print("  ✓ Confidence")
print("  ✓ Previous user preferences")

EXPLANATION SYSTEM ADDED

The app can now explain:
  ✓ Visual compatibility
  ✓ Mood
  ✓ Personality
  ✓ Occasion
  ✓ Confidence
  ✓ Previous user preferences


In [ ]:
APP_CODE = r'''
# ==================================================
# USER INPUT SECTION
# ==================================================

st.header("👕 Your Wardrobe")

st.write(
    "Upload the clothing items you want PsyStyle "
    "to use for recommendations."
)


uploaded_files = st.file_uploader(
    "Upload wardrobe images",
    type=[
        "jpg",
        "jpeg",
        "png"
    ],
    accept_multiple_files=True
)


if uploaded_files:

    st.subheader(
        f"Uploaded Items ({len(uploaded_files)})"
    )

    preview_columns = st.columns(
        min(len(uploaded_files), 5)
    )

    for index, uploaded_file in enumerate(
        uploaded_files
    ):

        image = Image.open(
            uploaded_file
        ).convert("RGB")

        with preview_columns[
            index % len(preview_columns)
        ]:

            st.image(
                image,
                caption=uploaded_file.name,
                use_container_width=True
            )


# ==================================================
# PSYCHOLOGICAL CONTEXT
# ==================================================

st.header("🧠 Your Psychological Context")


col1, col2 = st.columns(2)


with col1:

    mood = st.selectbox(
        "Mood",
        [
            "Happy",
            "Nervous",
            "Confident"
        ]
    )


    personality = st.selectbox(
        "Personality",
        [
            "Introvert",
            "Extrovert"
        ]
    )


with col2:

    occasion = st.selectbox(
        "Occasion",
        [
            "Interview",
            "Party",
            "College",
            "Wedding"
        ]
    )


    confidence = st.slider(
        "Confidence Level",
        min_value=1,
        max_value=10,
        value=7,
        step=1
    )

    num_outfits = st.slider(
    "Number of Outfits to Display",
    min_value=1,
    max_value=10,
    value=5,
    step=1
)


st.info(
    f"Selected context: "
    f"{mood} mood • "
    f"{personality} personality • "
    f"{occasion} • "
    f"Confidence {confidence}/10"
)


# ==================================================
# RECOMMEND BUTTON
# ==================================================

recommend_button = st.button(
    "✨ Recommend Outfit",
    type="primary",
    use_container_width=True
)
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("STREAMLIT INPUT INTERFACE ADDED")
print("=" * 55)

print("\nUser can now:")
print("  ✓ Upload multiple clothing images")
print("  ✓ Select mood")
print("  ✓ Select personality")
print("  ✓ Select occasion")
print("  ✓ Set confidence 1–10")
print("  ✓ Click Recommend Outfit")

STREAMLIT INPUT INTERFACE ADDED

User can now:
  ✓ Upload multiple clothing images
  ✓ Select mood
  ✓ Select personality
  ✓ Select occasion
  ✓ Set confidence 1–10
  ✓ Click Recommend Outfit


In [ ]:
APP_CODE = r'''
# ==================================================
# RECOMMENDATION PIPELINE
# ==================================================

if recommend_button:

    if not uploaded_files:

        st.warning(
            "Please upload at least two clothing "
            "images before generating an outfit."
        )

    else:

        with st.spinner(
            "Analyzing your wardrobe and generating "
            "recommendations..."
        ):

            wardrobe_items = []


            # ==========================================
            # PROCESS UPLOADED CLOTHING
            # ==========================================

            for uploaded_file in uploaded_files:

                image = Image.open(
                    uploaded_file
                ).convert("RGB")

                category, confidence_score, embedding = (
                    predict_clothing_item(image)
                )

                wardrobe_items.append({

                    "name":
                        uploaded_file.name,

                    "image":
                        image,

                    "category":
                        category,

                    "confidence":
                        confidence_score,

                    "embedding":
                        embedding
                })


            # ==========================================
            # SHOW CLOTHING CLASSIFICATIONS
            # ==========================================

            st.header(
                "🔍 Clothing Analysis"
            )

            analysis_columns = st.columns(
                min(len(wardrobe_items), 4)
            )

            for index, item in enumerate(
                wardrobe_items
            ):

                with analysis_columns[
                    index % len(analysis_columns)
                ]:

                    st.image(
                        item["image"],
                        caption=item["name"],
                        use_container_width=True
                    )

                    st.write(
                        f"**Category:** "
                        f"{item['category']}"
                    )

                    st.write(
                        f"**Confidence:** "
                        f"{item['confidence'] * 100:.2f}%"
                    )


            # ==========================================
            # GENERATE OUTFITS
            # ==========================================

            outfits = (
                generate_outfit_combinations(
                    wardrobe_items
                )
            )


            if not outfits:

                st.warning(
                    "No valid outfit combinations "
                    "could be generated from the "
                    "uploaded wardrobe."
                )

                st.info(
                    "Try uploading combinations such "
                    "as a top + bottom, or a dress/"
                    "one-piece outfit."
                )

            else:

                # ======================================
                # SCORE OUTFITS
                # ======================================

                user_preference = (
                    st.session_state
                    .user_preference_vector
                )

                scored_outfits = []

                for outfit in outfits:

                    result = score_outfit(

                        outfit=outfit,

                        mood=mood,

                        personality=personality,

                        occasion=occasion,

                        confidence=confidence,

                        user_preference=
                            user_preference
                    )

                    scored_outfits.append(
                        result
                    )


                # ======================================
                # RANK
                # ======================================

                ranked_outfits = rank_outfits(
                    scored_outfits
                )


                # Store recommendations in session
                st.session_state[
                    "ranked_outfits"
                ] = ranked_outfits

                st.session_state[
                    "recommendation_context"
                ] = {

                    "mood":
                        mood,

                    "personality":
                        personality,

                    "occasion":
                        occasion,

                    "confidence":
                        confidence
                }


                st.success(
                    f"{len(ranked_outfits)} outfit "
                    "combination(s) generated!"
                )
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("RECOMMENDATION PIPELINE CONNECTED")
print("=" * 55)

print("\nPipeline:")
print("  ✓ Uploaded images")
print("  ✓ ViT classification")
print("  ✓ 768-D embeddings")
print("  ✓ Outfit generation")
print("  ✓ Compatibility scoring")
print("  ✓ Psychology scoring")
print("  ✓ Preference scoring")
print("  ✓ Final scoring")
print("  ✓ Ranking")
print("  ✓ Session storage")

RECOMMENDATION PIPELINE CONNECTED

Pipeline:
  ✓ Uploaded images
  ✓ ViT classification
  ✓ 768-D embeddings
  ✓ Outfit generation
  ✓ Compatibility scoring
  ✓ Psychology scoring
  ✓ Preference scoring
  ✓ Final scoring
  ✓ Ranking
  ✓ Session storage


In [ ]:
APP_CODE = r'''
# ==================================================
# DISPLAY RECOMMENDATIONS
# ==================================================

if "ranked_outfits" in st.session_state:

    ranked_outfits = (
        st.session_state[
            "ranked_outfits"
        ]
    )

    context = (
        st.session_state[
            "recommendation_context"
        ]
    )


    st.header(
        "✨ Recommended Outfits"
    )


    # Show maximum 5 recommendations

    # Show the number of recommendations selected by the user

    top_outfits = ranked_outfits[:num_outfits]


    for rank, result in enumerate(
        top_outfits,
        start=1
    ):

        st.markdown("---")


        # ==========================================
        # OUTFIT TITLE
        # ==========================================

        st.subheader(
            f"🏆 Outfit #{rank}"
        )


        # ==========================================
        # OUTFIT IMAGES
        # ==========================================

        items = result["items"]

        image_columns = st.columns(
            len(items)
        )


        for index, item in enumerate(
            items
        ):

            with image_columns[index]:

                st.image(
                    item["image"],
                    caption=item["category"],
                    use_container_width=True
                )


        # ==========================================
        # CATEGORIES
        # ==========================================

        st.write(
            "**Items:** "
            +
            " + ".join(
                result["categories"]
            )
        )


        # ==========================================
        # SCORES
        # ==========================================

        score_col1, score_col2, score_col3, score_col4 = (
            st.columns(4)
        )


        with score_col1:

            st.metric(
                "Compatibility",
                f"{result['compatibility_score']:.2f}%"
            )


        with score_col2:

            st.metric(
                "Psychology",
                f"{result['psychology_score']:.2f}%"
            )


        with score_col3:

            st.metric(
                "Preference",
                f"{result['preference_score']:.2f}%"
            )


        with score_col4:

            st.metric(
                "Final Score",
                f"{result['final_score']:.2f}%"
            )


        # ==========================================
        # EXPLANATION
        # ==========================================

        st.write(
            "**Why this outfit?**"
        )


        explanations = (
            generate_outfit_explanation(

                result=result,

                mood=context["mood"],

                personality=
                    context["personality"],

                occasion=
                    context["occasion"],

                confidence=
                    context["confidence"]
            )
        )


        for explanation in explanations:

            st.write(
                f"✓ {explanation}"
            )


        # ==========================================
        # FEEDBACK BUTTONS
        # ==========================================

        button_col1, button_col2, button_col3 = (
            st.columns(3)
        )


        with button_col1:

            like_clicked = st.button(
                "❤️ Like",
                key=f"like_{rank}"
            )


        with button_col2:

            dislike_clicked = st.button(
                "👎 Dislike",
                key=f"dislike_{rank}"
            )


        with button_col3:

            skip_clicked = st.button(
                "— Skip",
                key=f"skip_{rank}"
            )


        # ==========================================
        # PROCESS FEEDBACK
        # ==========================================

        if like_clicked:

            process_user_feedback(
                result,
                "like"
            )

            st.session_state[
                "feedback_message"
            ] = (
                "❤️ Preference updated! "
                "Future recommendations will "
                "consider this choice."
            )

            st.rerun()


        if dislike_clicked:

            process_user_feedback(
                result,
                "dislike"
            )

            st.session_state[
                "feedback_message"
            ] = (
                "👎 Preference updated! "
                "Future recommendations will "
                "move away from this style."
            )

            st.rerun()


        if skip_clicked:

            st.session_state[
                "feedback_message"
            ] = (
                "— Skipped. No preference update "
                "was made."
            )

            st.rerun()


# ==================================================
# FEEDBACK MESSAGE
# ==================================================

if "feedback_message" in st.session_state:

    st.info(
        st.session_state[
            "feedback_message"
        ]
    )
'''

with open(APP_PATH, "a") as f:
    f.write("\n\n")
    f.write(APP_CODE)


print("=" * 55)
print("RECOMMENDATION DISPLAY ADDED")
print("=" * 55)

print("\nThe app can now display:")
print("  ✓ Top 5 ranked outfits")
print("  ✓ Outfit images")
print("  ✓ Clothing categories")
print("  ✓ Compatibility score")
print("  ✓ Psychology score")
print("  ✓ Preference score")
print("  ✓ Final score")
print("  ✓ Rule-based explanation")
print("  ✓ Like / Dislike / Skip feedback")

RECOMMENDATION DISPLAY ADDED

The app can now display:
  ✓ Top 5 ranked outfits
  ✓ Outfit images
  ✓ Clothing categories
  ✓ Compatibility score
  ✓ Psychology score
  ✓ Preference score
  ✓ Final score
  ✓ Rule-based explanation
  ✓ Like / Dislike / Skip feedback


In [ ]:
import ast
import os

print("=" * 55)
print("VALIDATING PSYSTYLE STREAMLIT APP")
print("=" * 55)

if not os.path.exists(APP_PATH):
    print("\n❌ app.py was not found.")

else:

    with open(APP_PATH, "r") as f:
        app_code = f.read()

    print("\nApp file found:")
    print(APP_PATH)

    print("\nApp size:")
    print(f"{len(app_code):,} characters")

    try:

        ast.parse(app_code)

        print("\n✓ Syntax check passed!")
        print("✓ app.py is valid Python.")
        print("✓ No training will occur during launch.")

    except SyntaxError as e:

        print("\n❌ Syntax error found.")

        print(
            f"Line: {e.lineno}"
        )

        print(
            f"Error: {e.msg}"
        )

        print(
            "\nPlease send me this output before "
            "launching Streamlit."
        )

VALIDATING PSYSTYLE STREAMLIT APP

App file found:
/content/drive/MyDrive/PsyStyle/streamlit/app.py

App size:
37,304 characters

✓ Syntax check passed!
✓ app.py is valid Python.
✓ No training will occur during launch.


In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 57.5 MB/s eta 0:00:00


In [ ]:
import subprocess
import time
import os

# Stop any previous Streamlit process
os.system("pkill -f 'streamlit run'")

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("=" * 55)
print("PSYSTYLE STREAMLIT SERVER STARTED")
print("=" * 55)

print("\nLocal server:")
print("http://localhost:8501")

print("\nProcess ID:")
print(process.pid)

print("\nNext step:")
print("We'll expose this server so you can open PsyStyle in your browser.")

PSYSTYLE STREAMLIT SERVER STARTED

Local server:
http://localhost:8501

Process ID:
43774

Next step:
We'll expose this server so you can open PsyStyle in your browser.


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3J5u3K8PShadnUo8wLjFb3kKzas_7h1MWNX5B9MbW4XCPiv8f"

ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)

print("ngrok authentication configured successfully.")

ngrok authentication configured successfully.


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(
    8501,
    "http"
)

print("=" * 55)
print("PSYSTYLE IS LIVE")
print("=" * 55)
print("\nOpen this URL in your browser:")
print(public_url)

PSYSTYLE IS LIVE

Open this URL in your browser:
NgrokTunnel: "https://vertebrae-defiant-squishier.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import ast

with open(APP_PATH, "r") as f:
    app_source = f.read()

ast.parse(app_source)

print("=" * 55)
print("✓ PsyStyle app syntax is valid")
print("=" * 55)

✓ PsyStyle app syntax is valid


In [ ]:
import subprocess
import time
import os

# Stop the old Streamlit server
os.system("pkill -f 'streamlit run'")

# Start the updated app
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("=" * 55)
print("✓ PSYSTYLE RESTARTED")
print("=" * 55)
print("\nThe updated app is running on port 8501.")

✓ PSYSTYLE RESTARTED

The updated app is running on port 8501.


In [ ]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=10
    )

    print("Status code:", response.status_code)

    if response.status_code == 200:
        print("✓ Streamlit is responding correctly.")
    else:
        print("⚠ Streamlit responded, but something may be wrong.")

except Exception as e:
    print("❌ Streamlit is NOT responding.")
    print("Error:", e)

❌ Streamlit is NOT responding.
Error: HTTPConnectionPool(host='127.0.0.1', port=8501): Read timed out. (read timeout=10)


In [ ]:
import subprocess

print("=" * 55)
print("STREAMLIT PROCESS CHECK")
print("=" * 55)

result = subprocess.run(
    ["bash", "-c", "ps aux | grep '[s]treamlit'"],
    capture_output=True,
    text=True
)

print(result.stdout)

print("\n" + "=" * 55)
print("RECENT STREAMLIT LOG")
print("=" * 55)

result = subprocess.run(
    ["bash", "-c", "tail -50 /tmp/streamlit.log 2>/dev/null || echo 'No log file found'"],
    capture_output=True,
    text=True
)

print(result.stdout)

STREAMLIT PROCESS CHECK
root       43774  1.3 11.6 13767628 1554556 ?    Sl   12:46   0:41 /usr/bin/python3 /usr/local/bin/streamlit run /content/drive/MyDrive/PsyStyle/streamlit/app.py --server.port 8501 --server.address 0.0.0.0
root       55466  0.9  0.0      0     0 ?        Z    13:32   0:02 [streamlit] <defunct>


RECENT STREAMLIT LOG
No log file found



In [ ]:
import subprocess
import time
import os

# Stop all existing Streamlit processes
os.system("pkill -9 -f 'streamlit run'")

time.sleep(2)

# Start a completely fresh Streamlit server
log_file = open("/tmp/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("=" * 55)
print("✓ FRESH STREAMLIT SERVER STARTED")
print("=" * 55)
print("Process ID:", process.pid)

print("\nRecent server output:")
log_file.flush()

with open("/tmp/streamlit.log", "r") as f:
    print(f.read()[-3000:])

✓ FRESH STREAMLIT SERVER STARTED
Process ID: 56799

Recent server output:


2026-09-28 13:37:21.881 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.96.146:8501




In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8501",
    timeout=30
)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("✓ Streamlit is responding correctly.")
    print("✓ The app itself is working.")
else:
    print("⚠ Streamlit responded with an unexpected status.")

Status code: 200
✓ Streamlit is responding correctly.
✓ The app itself is working.


In [ ]:
from pyngrok import ngrok

# Close any old tunnels
ngrok.kill()

# Create a new tunnel to the working Streamlit server
public_url = ngrok.connect(
    8501,
    "http"
)

print("=" * 55)
print("✓ PSYSTYLE IS LIVE")
print("=" * 55)
print("\nOpen this NEW URL in your browser:")
print(public_url)

✓ PSYSTYLE IS LIVE

Open this NEW URL in your browser:
NgrokTunnel: "https://vertebrae-defiant-squishier.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
with open(APP_PATH, "r") as f:
    lines = f.readlines()

print("=" * 55)
print("FILE UPLOADER OCCURRENCES")
print("=" * 55)

for i, line in enumerate(lines, start=1):
    if "st.file_uploader" in line:
        print(f"Line {i}: {line.strip()}")

FILE UPLOADER OCCURRENCES
Line 1381: uploaded_files = st.file_uploader(
Line 1958: uploaded_files = st.file_uploader(


In [ ]:
with open(APP_PATH, "r") as f:
    lines = f.readlines()

print("=" * 55)
print("AROUND FIRST INTERFACE")
print("=" * 55)

for i in range(1350, 1410):
    print(f"{i+1}: {lines[i].rstrip()}")

print("\n" + "=" * 55)
print("AROUND DUPLICATE INTERFACE")
print("=" * 55)

for i in range(1925, 1995):
    print(f"{i+1}: {lines[i].rstrip()}")

AROUND FIRST INTERFACE
1351: 
1352:         explanations.append(
1353:             "The outfit differs from your "
1354:             "previous preferences."
1355:         )
1356: 
1357:     else:
1358: 
1359:         explanations.append(
1360:             "There is currently limited preference "
1361:             "feedback from the user."
1362:         )
1363: 
1364: 
1365:     return explanations
1366: 
1367: 
1368: 
1369: # ==================================================
1370: # USER INPUT SECTION
1371: # ==================================================
1372: 
1373: st.header("👕 Your Wardrobe")
1374: 
1375: st.write(
1376:     "Upload the clothing items you want PsyStyle "
1377:     "to use for recommendations."
1378: )
1379: 
1380: 
1381: uploaded_files = st.file_uploader(
1382:     "Upload wardrobe images",
1383:     type=[
1384:         "jpg",
1385:         "jpeg",
1386:         "png"
1387:     ],
1388:     accept_multiple_files=True
1389: )
1390: 
1391: 
1392: if uploaded_fi

In [ ]:
with open(APP_PATH, "r") as f:
    lines = f.readlines()

# The original app ends before the second duplicated
# "USER INPUT SECTION" at line 1947.
cleaned_lines = lines[:1945]

with open(APP_PATH, "w") as f:
    f.writelines(cleaned_lines)

print("=" * 55)
print("✓ DUPLICATE INTERFACE REMOVED")
print("=" * 55)
print("Kept the original interface.")
print("Removed the duplicate section starting at line 1946.")

✓ DUPLICATE INTERFACE REMOVED
Kept the original interface.
Removed the duplicate section starting at line 1946.


In [ ]:
import ast

with open(APP_PATH, "r") as f:
    app_source = f.read()

ast.parse(app_source)

print("=" * 55)
print("✓ PSYSTYLE APP SYNTAX IS VALID")
print("=" * 55)

print("\nChecking file uploader count...")

count = app_source.count("st.file_uploader")

print(f"File uploader occurrences: {count}")

if count == 1:
    print("✓ Exactly one file uploader found.")
else:
    print("⚠ Unexpected number of file uploaders.")

✓ PSYSTYLE APP SYNTAX IS VALID

Checking file uploader count...
File uploader occurrences: 1
✓ Exactly one file uploader found.


In [ ]:
import subprocess
import time
import os

# Stop the current Streamlit server
os.system("pkill -9 -f 'streamlit run'")

time.sleep(2)

# Start the cleaned app
log_file = open("/tmp/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("=" * 55)
print("✓ PSYSTYLE RESTARTED WITH FIXED APP")
print("=" * 55)

with open("/tmp/streamlit.log", "r") as f:
    print(f.read()[-2000:])

✓ PSYSTYLE RESTARTED WITH FIXED APP


2026-09-28 13:42:21.065 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.96.146:8501




In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8501",
    timeout=30
)

print("=" * 55)
print("LOCAL STREAMLIT CHECK")
print("=" * 55)
print("Status code:", response.status_code)

if response.status_code == 200:
    print("✓ Streamlit is responding.")
    print("✓ Duplicate uploader issue is fixed.")
else:
    print("⚠ Unexpected response.")

LOCAL STREAMLIT CHECK
Status code: 200
✓ Streamlit is responding.
✓ Duplicate uploader issue is fixed.


In [ ]:
from pyngrok import ngrok

# Close all existing tunnels
ngrok.kill()

# Create a fresh tunnel to the working Streamlit server
public_url = ngrok.connect(
    "http://127.0.0.1:8501"
)

print("=" * 55)
print("✓ PSYSTYLE IS LIVE")
print("=" * 55)
print("\nNEW PUBLIC URL:")
print(public_url)

✓ PSYSTYLE IS LIVE

NEW PUBLIC URL:
NgrokTunnel: "https://vertebrae-defiant-squishier.ngrok-free.dev" -> "http://127.0.0.1:8501"


In [ ]:
with open(APP_PATH, "r") as f:
    lines = f.readlines()

print("=" * 55)
print("NUMBER OF OUTFITS CODE")
print("=" * 55)

found = False

for i, line in enumerate(lines, start=1):
    if "num_outfits" in line or "Number of Outfits" in line:
        print(f"{i}: {line.rstrip()}")
        found = True

if not found:
    print("❌ No num_outfits code found in app.py")

NUMBER OF OUTFITS CODE
❌ No num_outfits code found in app.py


In [ ]:
with open(APP_PATH, "r") as f:
    app_source = f.read()

old_code = '''confidence = st.slider(
    "Confidence Level",
    min_value=1,
    max_value=10,
    value=5
)'''

new_code = '''confidence = st.slider(
    "Confidence Level",
    min_value=1,
    max_value=10,
    value=5
)

num_outfits = st.slider(
    "Number of Outfits to Display",
    min_value=1,
    max_value=10,
    value=5,
    step=1
)'''

if old_code not in app_source:
    print("❌ Could not find the confidence slider.")
else:
    app_source = app_source.replace(
        old_code,
        new_code,
        1
    )

    app_source = app_source.replace(
        "top_outfits = ranked_outfits[:5]",
        "top_outfits = ranked_outfits[:num_outfits]",
        1
    )

    with open(APP_PATH, "w") as f:
        f.write(app_source)

    print("=" * 55)
    print("✓ OUTFIT COUNT CONTROL ADDED")
    print("=" * 55)
    print("✓ Slider: 1 to 10 outfits")
    print("✓ Default: 5 outfits")
    print("✓ Recommendation display now uses num_outfits")

❌ Could not find the confidence slider.


In [ ]:
with open(APP_PATH, "r") as f:
    lines = f.readlines()

print("=" * 55)
print("CURRENT USER INPUT SECTION")
print("=" * 55)

for i, line in enumerate(lines, start=1):
    if "# USER INPUT SECTION" in line:
        start = i
        break
else:
    start = 1

for i in range(start, min(start + 180, len(lines) + 1)):
    print(f"{i}: {lines[i-1].rstrip()}")

CURRENT USER INPUT SECTION
1370: # USER INPUT SECTION
1371: # ==================================================
1372: 
1373: st.header("👕 Your Wardrobe")
1374: 
1375: st.write(
1376:     "Upload the clothing items you want PsyStyle "
1377:     "to use for recommendations."
1378: )
1379: 
1380: 
1381: uploaded_files = st.file_uploader(
1382:     "Upload wardrobe images",
1383:     type=[
1384:         "jpg",
1385:         "jpeg",
1386:         "png"
1387:     ],
1388:     accept_multiple_files=True
1389: )
1390: 
1391: 
1392: if uploaded_files:
1393: 
1394:     st.subheader(
1395:         f"Uploaded Items ({len(uploaded_files)})"
1396:     )
1397: 
1398:     preview_columns = st.columns(
1399:         min(len(uploaded_files), 5)
1400:     )
1401: 
1402:     for index, uploaded_file in enumerate(
1403:         uploaded_files
1404:     ):
1405: 
1406:         image = Image.open(
1407:             uploaded_file
1408:         ).convert("RGB")
1409: 
1410:         with preview_columns[
1411

In [ ]:
with open(APP_PATH, "r") as f:
    app_source = f.read()

# Add the outfit-count slider after the existing confidence slider
old_confidence = '''    confidence = st.slider(
        "Confidence Level",
        min_value=1,
        max_value=10,
        value=7,
        step=1
    )'''

new_confidence = '''    confidence = st.slider(
        "Confidence Level",
        min_value=1,
        max_value=10,
        value=7,
        step=1
    )

    num_outfits = st.slider(
        "Number of Outfits to Display",
        min_value=1,
        max_value=10,
        value=5,
        step=1
    )'''

if old_confidence not in app_source:
    print("❌ Confidence slider pattern not found.")
else:
    app_source = app_source.replace(
        old_confidence,
        new_confidence,
        1
    )

    # Make the recommendation display use the selected number
    if "top_outfits = ranked_outfits[:5]" in app_source:
        app_source = app_source.replace(
            "top_outfits = ranked_outfits[:5]",
            "top_outfits = ranked_outfits[:num_outfits]",
            1
        )

    with open(APP_PATH, "w") as f:
        f.write(app_source)

    print("=" * 55)
    print("✓ OUTFIT COUNT CONTROL ADDED")
    print("=" * 55)
    print("✓ Slider range: 1–10")
    print("✓ Default: 5")
    print("✓ Recommendation display uses selected count")

✓ OUTFIT COUNT CONTROL ADDED
✓ Slider range: 1–10
✓ Default: 5
✓ Recommendation display uses selected count


In [ ]:
with open(APP_PATH, "r") as f:
    app_source = f.read()

print("=" * 55)
print("VERIFYING OUTFIT COUNT CONTROL")
print("=" * 55)

print("\n1. Checking slider:")
if "Number of Outfits to Display" in app_source:
    print("✓ Outfit count slider found.")
else:
    print("❌ Outfit count slider NOT found.")

print("\n2. Checking recommendation limit:")
if "ranked_outfits[:num_outfits]" in app_source:
    print("✓ Recommendations use selected outfit count.")
else:
    print("❌ Recommendation limit NOT updated.")

print("\n3. Checking syntax:")
import ast
ast.parse(app_source)
print("✓ App syntax is valid.")

VERIFYING OUTFIT COUNT CONTROL

1. Checking slider:
✓ Outfit count slider found.

2. Checking recommendation limit:
✓ Recommendations use selected outfit count.

3. Checking syntax:
✓ App syntax is valid.


In [ ]:
import subprocess
import time
import os

os.system("pkill -9 -f 'streamlit run'")

time.sleep(2)

log_file = open("/tmp/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("=" * 55)
print("✓ PSYSTYLE RESTARTED")
print("=" * 55)

with open("/tmp/streamlit.log", "r") as f:
    print(f.read()[-2000:])

✓ PSYSTYLE RESTARTED


2026-09-28 13:49:30.944 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.96.146:8501




In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8501",
    timeout=30
)

print("=" * 55)
print("LOCAL APP CHECK")
print("=" * 55)
print("Status code:", response.status_code)

if response.status_code == 200:
    print("✓ Streamlit is responding.")
    print("✓ Updated app is running.")
else:
    print("⚠ Unexpected response.")

LOCAL APP CHECK
Status code: 200
✓ Streamlit is responding.
✓ Updated app is running.


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(
    "http://127.0.0.1:8501"
)

print("=" * 55)
print("✓ PSYSTYLE IS LIVE")
print("=" * 55)
print("\nNEW URL:")
print(public_url)

✓ PSYSTYLE IS LIVE

NEW URL:
NgrokTunnel: "https://vertebrae-defiant-squishier.ngrok-free.dev" -> "http://127.0.0.1:8501"


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers timm scikit-learn pandas numpy pillow matplotlib tqdm streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 58.8 MB/s eta 0:00:00


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/PsyStyle"

folders = [
    "models",
    "models/vit",
    "models/compatibility",
    "models/preference",
    "models/fusion",
    "data",
    "data/deepfashion1",
    "data/polyvore",
    "data/psychology",
    "wardrobe",
    "outputs",
    "streamlit"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("✓ PsyStyle paths ready")

✓ PsyStyle paths ready


In [4]:
!streamlit run "/content/drive/MyDrive/PsyStyle/streamlit/app.py" --server.port 8501 > /tmp/streamlit.log 2>&1 &

In [5]:
# ============================================================
# 🚀 PSYSTYLE — START STREAMLIT AFTER RUNTIME DISCONNECT
# ============================================================

import os
import time
import socket
import subprocess

APP_PATH = "/content/drive/MyDrive/PsyStyle/streamlit/app.py"

print("=" * 60)
print("STARTING PSYSTYLE")
print("=" * 60)

# Stop any old Streamlit process if one exists
os.system("pkill -f 'streamlit run' >/dev/null 2>&1")

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until port 8501 is actually ready
print("Waiting for Streamlit to start...")

ready = False

for i in range(60):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)

    try:
        sock.connect(("127.0.0.1", 8501))
        ready = True
        sock.close()
        break
    except:
        sock.close()
        time.sleep(1)

if ready:
    print("✓ Streamlit is running")
    print("✓ Port 8501 is ready")
    print("✓ PsyStyle app is ready")
else:
    print("❌ Streamlit did not start within 60 seconds")
    print("Check the Streamlit logs if this happens.")

STARTING PSYSTYLE
Waiting for Streamlit to start...
✓ Streamlit is running
✓ Port 8501 is ready
✓ PsyStyle app is ready


In [6]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3J5u3K8PShadnUo8wLjFb3kKzas_7h1MWNX5B9MbW4XCPiv8f"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✓ ngrok authentication configured")

✓ ngrok authentication configured


In [7]:
from pyngrok import ngrok

print("=" * 60)
print("CREATING PSYSTYLE PUBLIC URL")
print("=" * 60)

ngrok.kill()

public_url = ngrok.connect("http://127.0.0.1:8501")

print("✓ PsyStyle is live!")
print()
print("PUBLIC URL:")
print(public_url)

CREATING PSYSTYLE PUBLIC URL
✓ PsyStyle is live!

PUBLIC URL:
NgrokTunnel: "https://vertebrae-defiant-squishier.ngrok-free.dev" -> "http://127.0.0.1:8501"
